# AD-Textvergleich – PRO (MDR ↔ KI) · Kurzbeschreibung

Dieses Notebook vergleicht zwei Audiodeskriptions-Texte **satzweise** (MDR-Referenz vs. KI-Text), richtet die Sätze automatisch **einander zu** und bewertet die **inhaltliche Nähe**.

**Was es macht**
- Segmentiert, tokenisiert und **aligned** die Sätze (TF-IDF + Cosine-Ähnlichkeit).
- Bewertet Qualität mit **BLEU-1..4**, **ROUGE-1/2/L**, optional **SBERT-Cosine** und **BERTScore F1**.
- Ergänzt **RAGAS-Kennzahlen** (semantic_similarity, answer_similarity, faithfulness, answer_relevancy, coverage, conciseness, fluency ≈ FRE) – mit **Fallbacks**, falls RAGAS nicht verfügbar ist.
- Erfasst **Heuristiken**: Farbdetails & Bewegungsverben (lemma-basiert, mit spaCy-Fallback).
- Misst **Wort- und Silbenlängen** je Segment sowie **Lesbarkeit** (Flesch–Amstad).
- Zeigt Ergebnisse in **Tabs**: *Text*, *Qualität*, *ROUGE-Details*, *Länge & Abdeckung*, *Alignment*, *RAGAS*, *Alle*, *Einstellungen* – mit **Ampel-Darstellung**.
- Visualisiert die Verteilungen und Beziehungen in **10 einheitlichen Charts** (je 2 nebeneinander).
- Fasst alles in einer **automatischen Interpretation** zusammen.
- Liefert **MDR-Rubrics (0–100)** für Stil/Konzision, Lesehärte, visuelle Klarheit, Handlungsführung und Inhaltsdeckung inkl. **Gesamtnote**.

**Hinweise**
- Schwellen für die Ampel sind im Notebook **konfigurierbar** (Tab *Einstellungen* → *Ampel-Schwellen*).  
- **RAGAS-Grenzen** sind aktuell im Code (`THRESHOLDS`) hinterlegt, die **Kernmetriken** können über Slider angepasst werden.  
- **SBERT**, **BERTScore** und **spaCy** werden nur verwendet, **wenn installiert** – sonst laufen die übrigen Auswertungen weiter.  
- Eingabe: Laden Sie den **MDR-Referenztext** und den **KI-Text** über die Upload-Felder und klicken Sie anschließend auf **Berechnen**.


In [1]:
# OPTIONAL: Installieren, falls erforderlich
# %pip install -q sentence-transformers bert-score matplotlib pandas numpy

# !pip install ragas nest_asyncio sentence-transformers langchain-openai

# %pip install -U spacy
# !python -m spacy download de_core_news_sm

# import sys
# !{sys.executable} -m pip install -U pyphen


# Kennzahlen – Definition, Berechnung, Interpretation & Empfehlungen

Diese Seite erklärt **alle Metriken** unseres AD-Textvergleichs (klassische NLP-Scores, **Silben-Maße**, **RAGAS**, **MDR-Rubrics**) in **einfachen Worten** und gibt **konkrete Handlungsempfehlungen**.

> **Hinweise**
> - **Ampelschwellen** sind im Notebook über `THRESHOLDS` konfigurierbar (UI-Slider für die **Kernmetriken**; RAGAS-Grenzen derzeit im Code).
> - **SBERT**/**BERTScore** werden nur berechnet, wenn die Pakete installiert sind.
> - **RAGAS** wird segmentweise auf *Ref (MDR) ↔ Hyp (KI)* angewendet; höher = besser.  
>   Falls die Bibliothek (oder Teilmetriken) fehlen, verwenden wir **RAGAS-ähnliche Fallbacks** aus ROUGE/SBERT/BERT/FRE.

---

## Übersichtstabelle (Kernmetriken)

| Kennzahl | Idee in einem Satz | Berechnung (vereinfacht) | Skala & Aussage | Ampel/Daumenregel | Bei **niedrigen** Werten | Bei **hohen** Werten | Grenzen |
|---|---|---|---|---|---|---|---|
| **Cosine (TF-IDF)** | Wortlaut-Nähe über Schlüsselwörter | TF-IDF-Vektoren je Segment → Kosinus | 0–1 (1 = sehr ähnlich) | **<0.60** schwach · **0.60–0.75** ok · **>0.75** gut | Terminologie ergänzen, Alignment prüfen | Terminologie/Stil konsistent | Paraphrasen werden unterschätzt |
| **SBERT Cosine** | Semantische Nähe trotz Paraphrase | Satz-Embeddings (all-MiniLM) → Kosinus | 0–1; Bedeutungsähnlichkeit | **<0.75** schwach · **0.75–0.85** ok · **>0.85** gut | Inhalte/Lücken schließen, Fakten explizit nennen | Inhalt deckungsgleich trotz anderer Form | Paket `sentence-transformers` nötig |
| **BERTScore F1** | Bedeutungs-Overlap auf Tokenebene | mBERT-Embeddings: Präzision/Recall → F1 | 0–1; robust ggü. Wortreihenfolge | **<0.85** prüfen · **0.85–0.90** ok · **>0.90** top | fehlende Details nachtragen | sehr gute Abdeckung | Paket `bert-score` nötig |
| **ROUGE-L F1** | Abdeckung inkl. **Reihenfolge** | LCS; aus P/R → F1 | 0–1; Strukturtreue | **<0.45** schwach · **0.45–0.60** ok · **>0.60** gut | Reihenfolge & Vollständigkeit angleichen | gute Deckung & Reihenfolge | paraphrasen-sensitiv |
| **ROUGE-1/2 P/R/F1** | n-Gram Überlappung | n=1/2 Overlap → P, R, F1 | 0–1 | – | **P ≫ R:** KI zu knapp → ergänzen · **R ≫ P:** KI zu lang → straffen | – | Wortlaut-fokussiert |
| **BLEU-1..4** | Klassische Wortlauttreue | geom. Mittel Präzisionen + Kürzungsstrafe | 0–1 | **BLEU-4:** **<0.15** niedrig · **0.15–0.25** ok · **>0.25** gut | ist BERT/ROUGE gut? → Paraphrase ok; sonst Terminologie angleichen | hohe Formulierungstreue | paraphrasen-unfreundlich |
| **Wörter Ref/Hyp** | Segmentlänge (Tokens) | einfacher Regex-Tokenizer | Anzahl | – | **Hyp ≪ Ref:** Details fehlen | **Hyp ≫ Ref:** straffen | zählt Zahlen/Bindestriche mit |
| **`len_ratio` (Hyp/Ref)** | Längenverhältnis Wörter | `Wörter_Hyp / Wörter_Ref` | Ideal ≈ **1.0** | **0.85–1.15** gut · **<0.9** zu kurz · **>1.1** zu lang | Details/Farben/Bewegungen ergänzen | Redundanz entfernen, Sätze kürzen | misst nur Länge |

### Zusatz: **Silben-Maße (Tempo/Lesedauer)**

| Kennzahl | Idee | Berechnung | Skala | Regeln | Maßnahmen |
|---|---|---|---|---|---|
| **Silben Ref/Hyp** | grobe Sprechzeit-Indikation | heurist. Silbenzählung je Segment | Anzahl | – | **Hyp ≪ Ref:** evtl. zu knapp/schnell · **Hyp ≫ Ref:** evtl. zu langsam/ausführlich |
| **`syl_ratio` (Hyp/Ref)** | Längenverhältnis **Silben** | `Silben_Hyp / Silben_Ref` | Ideal ≈ **1.0** | **0.85–1.15** gut | wie `len_ratio`, aber tempo-sensitiver |

---

## RAGAS-Metriken (segmentweise)

| Kennzahl | Idee in einem Satz | Was misst sie hier? | Skala | Ampel (Default) | Maßnahmen bei **niedrig** |
|---|---|---|---|---|---|
| **semantic_similarity** | Bedeutungsnähe gesamt | Embedding-Nähe Ref↔Hyp | 0–1 | **0.70/0.85** | zentrale Aussagen angleichen |
| **answer_similarity** | Oberflächen-/Formulierungsnähe | semantische Nähe „Antwort ↔ Referenz“ | 0–1 | **0.70/0.85** | Terminologie/Benennungen vereinheitlichen |
| **faithfulness** | Fakten-Treue | keine Halluzinationen ggü. Ref | 0–1 | **0.60/0.75** | an Ref halten, falsche Details entfernen |
| **answer_relevancy** | Relevanz | passt Hyp inhaltlich zur Ref? | 0–1 | **0.60/0.75** | Off-Topic kürzen, Fokus schärfen |
| **coverage** | Abdeckung | wie viel der Ref deckt Hyp ab? | 0–1 | **0.50/0.70** | fehlende Elemente ergänzen |
| **conciseness** | Prägnanz | unnötiges in Hyp? (höher = knapper) | 0–1 | **0.60/0.80** | Füllwörter/Redundanz streichen |
| **fluency (FRE)** | Lesbarkeit | Flesch-Amstad (normiert 0–1) | 0–1 | **0.60/0.75** | Sätze vereinfachen, Aktiv, konkrete Verben |

> **Interpretationshilfe:**  
> - **semantic/answer_similarity hoch**, **coverage/faithfulness niedrig** → schön formuliert, aber **Inhalt unvollständig** oder **ungenau**.  
> - **coverage hoch**, **conciseness niedrig** → **zu breit** → kürzen.

---

## MDR-Rubrics (0–100, höher besser)

| Rubrik | Idee | (vereinfachte) Herleitung |
|---|---|---|
| **Stil/Konzision (Wörter)** | Nähe der Längen | Score aus `|1 − len_ratio|` |
| **Stil/Konzision (Silben)** | Tempo/Lesedauer-Nähe | Score aus `|1 − syl_ratio|` |
| **Lesehärte (FRE, Hyp)** | Verständlichkeit | Flesch-Amstad (normiert) |
| **Visuelle Klarheit** | Farbinfos vollständig | kleiner Gap **Farbe Ref − Hyp** besser |
| **Handlungsführung** | Bewegungen benannt | kleiner Gap **Bewegung Ref − Hyp** besser |
| **Inhaltsdeckung/Kohärenz** | Inhalt & Reihenfolge | Mittel aus **ROUGE-L F1** und **SBERT** |

> **Gesamtnote**: gewichtetes Mittel der sechs Rubriken (Standard: gleich gewichtet).

---

## Ableitungen & Entscheidungsregeln (praxisnah)

- **Paraphrase erkannt:** *SBERT/BERTScore hoch* **und** *BLEU niedrig* → Inhalt ok, Wortlaut anders. **Terminologie nur dort angleichen**, wo sie fachlich nötig ist.  
- **Abdeckung vs. Präzision:**  
  - **ROUGE-L R niedrig, P hoch** / **RAGAS coverage niedrig** → **fehlende Elemente ergänzen**.  
  - **ROUGE-L R hoch, P niedrig** / **RAGAS conciseness niedrig** → **kürzen**, Fokus schärfen.  
- **Länge & Tempo:**  
  - **`len_ratio` < 0.9** oder **`syl_ratio` < 0.9** → Details/Bewegungen/Farben ergänzen, ggf. Sätze bündeln.  
  - **`len_ratio` > 1.1** oder **`syl_ratio` > 1.1** → Redundanz entfernen, Nebensätze vereinfachen.  
- **Visuelle Qualität:**  
  - **Farbdetail Hyp ≪ Ref** → explizite Farbangaben (Kleid, Mappen, Licht) nachtragen.  
  - **Bewegung Hyp ≪ Ref** → klare Verben („betritt“, „dreht sich“, „setzt sich“) nutzen.

---

## Streuung & Stabilität (CV %)

**CV %** = *100 · (Standardabweichung / Mittelwert)* je Kennzahl (segmentweise).  
- **≤ 20 %** sehr stabil · **≤ 40 %** moderat · **> 40 %** volatil → Qualität schwankt.  
**Vorgehen bei hoher Streuung:** Schwachstellenliste bilden (ROUGE-L/BERTScore/coverage niedrig), Alignment prüfen, gezielt nachschärfen.

---

## Ampelschwellen (Defaults im Notebook)

- **Cosine (TF-IDF):** gelb **0.60**, grün **0.75**  
- **SBERT Cosine:** gelb **0.75**, grün **0.85**  
- **BERTScore F1:** gelb **0.85**, grün **0.90**  
- **ROUGE-L F1:** gelb **0.45**, grün **0.60**  
- **BLEU-4:** gelb **0.15**, grün **0.25**  
- **`len_ratio`, `syl_ratio`:** gut **0.85–1.15**  
- **RAGAS** *(höher = besser; Fallbacks möglich)*:  
  - **semantic_similarity:** **0.70/0.85**  
  - **answer_similarity:** **0.70/0.85**  
  - **faithfulness:** **0.60/0.75**  
  - **answer_relevancy:** **0.60/0.75**  
  - **coverage:** **0.50/0.70**  
  - **conciseness:** **0.60/0.80**  
  - **fluency (FRE):** **0.60/0.75**

> Diese Schwellen sind **heuristisch**. Passe sie bei Bedarf in `THRESHOLDS` an **Domäne**, **Sprechtakt** und **Stilvorgaben** an.  
> (UI-Slider existieren für die Kernmetriken; RAGAS-Grenzen aktuell per Code.)


In [20]:
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# - Segmentierung & Alignment (TF-IDF + Cosine)
# - BLEU-1..4, ROUGE-1/2/L
# - SBERT Cosine (optional)
# - BERTScore F1 (optional)
# - Heuristiken (Farben, Bewegungsverben), optional lemmatisiert (spaCy)
# - Silben, Flesch–Amstad Lesbarkeit (DE)
# - RAGAS: faithfulness / answer_relevancy / semantic_similarity / coverage / conciseness / (fluency≈FRE) – mit Fallbacks
# - Tabs: Text / Qualität / ROUGE-Details / Länge & Abdeckung / Alignment / RAGAS / Alle / Einstellungen
# - 10 Charts, einheitliche Größe, je 2 pro Figure
# - Vollständiger Auto-Report inkl. MDR Rubrics (Stil/Lesehärte/…)
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

import re, math, os
from collections import Counter
from typing import List, Tuple, Dict, Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown, Javascript
import warnings, logging
import ipywidgets as W
from datetime import datetime
from html import escape

# --- Warnings & Logging ruhigstellen ---
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=".*encoder_attention_mask.*")
for name in ["transformers", "bertscore", "sentence_transformers", "torch"]:
    logging.getLogger(name).setLevel(logging.ERROR)

display(HTML("""
<style>
/* ── Global: Scrollleisten & Basics ─────────────────────────────── */
.jupyter-widgets .widget-box,
.jupyter-widgets .widget-gridbox,
.jupyter-widgets .widget-hbox,
.jupyter-widgets .widget-vbox,
.jupyter-widgets .widget-file-upload,
.jupyter-widgets .widget-progress,
.jp-OutputArea-output { overflow-x: hidden !important; }

/* ── Tabs: sticky + moderner Look ───────────────────────────────── */
.widget-tab .p-TabBar {
  position: sticky; top: 0; z-index: 1000; background:#fff;
  border-bottom: 1px solid #e6e9ef;
}
.widget-tab .p-TabBar-content { padding: 2px 4px 0 4px; }
.widget-tab .p-TabBar-tab {
  margin-right: 6px; padding: 6px 12px;
  border: 1px solid #d7dbe0; border-bottom: none;
  border-radius: 10px 10px 0 0; background:#f7f9fc; color:#222;
  transition: filter .15s ease, box-shadow .15s ease;
}
.widget-tab .p-TabBar-tab:hover { filter: brightness(0.98); }
.widget-tab .p-TabBar-tab.p-mod-current {
  font-weight: 600; box-shadow: 0 -1px 0 0 #9aa4b2 inset;
}

/* ── Accordion hübscher ─────────────────────────────────────────── */
.jp-Accordion-header, .p-Accordion-header {
  background:#f7f9fc !important; font-weight:600;
  border-radius:6px; padding:6px 10px; margin-top:6px;
  border:1px solid #e3e7ec;
}
.jp-Accordion-content, .p-Accordion-content {
  background:#ffffff; border:1px solid #e3e7ec; border-top:none;
  border-radius:0 0 6px 6px; padding:10px 12px;
}

/* ── Tabs-Container: EINZIGER Scroll-Container ──────────────────── */
.ad-tabs{
  max-height:72vh;
  overflow-y:auto !important;
  overflow-x:hidden !important;
  position:relative;
  overscroll-behavior:contain;
}
/* Tab-Leiste klebt im scrollenden Container */
.ad-tabs .p-TabBar{
  position:sticky; top:0; z-index:1001;
  background:#fff; border-bottom:1px solid #e6e9ef;
}
/* Panel selbst nicht scrollen (sonst doppelte Scrollbars) */
.ad-tabs .p-StackedPanel{ overflow:visible !important; }

/* ── Grid-Ansicht (Texttab) ─────────────────────────────────────── */
.adgrid table { width:100%; border-collapse:separate; border-spacing:0 6px; }
.adgrid th { text-align:left; font-weight:600; padding:6px 8px; }
.adgrid td { vertical-align:top; padding:6px 8px; line-height:1.35; }
.adgrid tr:nth-child(odd) td { background:#f0f7ff; }

/* ── Pandas-Table Basics (falls ohne Inline-Styles) ─────────────── */
table.dataframe { width:100%; border-collapse:collapse; }
table.dataframe th, table.dataframe td { padding:6px 8px; font-size:12px; }
table.dataframe tbody tr:nth-child(odd) { background:#f7f9fc; }

/* ── Pandas-Styler in Tabs: NUR .ad-tabs scrollt ────────────────── */
.ad-tabs table.dataframe{
  border-collapse: separate !important;  /* nötig für sticky */
  border-spacing: 0;
}
/* Sticky-Header relativ zum .ad-tabs-Scrollcontainer */
.ad-tabs table.dataframe thead{
  position: sticky;
  top: 0;                 /* klebt oben im EINEN Scroll-Container */
  z-index: 50;
  background: #f7f9fc !important;
}
.ad-tabs table.dataframe thead th{
  position: sticky;
  top: 0;
  z-index: 51;
  background: #f7f9fc !important;
  border-bottom: 2px solid #e0e0e0;
  box-shadow: 0 2px 2px rgba(0,0,0,0.05);
}
/* minimaler Abstand unter dem Header */
.ad-tabs table.dataframe tbody tr:first-child td{ padding-top: 4px; }

/* Styler-Wrapper entwaffnen → keine eigene Scrollbar */
.ad-tabs div[id^="T_"]{
  overflow: visible !important;
  max-height: none !important;
  height: auto !important;
}
.ad-tabs div[id^="T_"] > div{
  overflow: visible !important;
}

/* ── Optionaler Offset, falls Header unter die Tabbar rutscht ─────
   Aktivieren: Kommentar entfernen und px-Wert setzen
:root { --ad-tabbar-h: 0px; }  /* z.B. 36px */
.ad-tabs table.dataframe thead,
.ad-tabs table.dataframe thead th{ top: var(--ad-tabbar-h); }
------------------------------------------------------------------- */
</style>
"""))

def sticky_styler(styler_obj) -> str:
    """
    Nimmt einen pandas Styler entgegen und liefert HTML,
    eingebettet in einen Wrapper, der sich mit deinem CSS gut verträgt.
    """
    try:
        html = styler_obj.to_html()
    except TypeError:
        # ältere pandas-Versionen erwarten keine named args
        html = styler_obj.to_html()
    # der Wrapper verhindert eigene Scrollbars des Styler-DIVs
    return f"<div style='overflow:visible; max-width:100%' class='ad-styler-wrap'>{html}</div>"

# =======================
# ⚙️ Konfiguration & Umgebungs-Check
# =======================
CONFIG = {
    "LANG": "de",
    "USE_SPACY": True,
    "USE_SBERT": True,
    "USE_BERTSCORE": True,
    "USE_RAGAS": True,          # versucht echte ragas-Metriken; sonst Fallback
    "SYLLABLE_METHOD": "heuristic_de"
}

# Default-Flags (werden im Panel gesetzt)
__COMPUTE_SBERT__     = True
__COMPUTE_BERTSCORE__ = True
__SHOW_RAGAS__        = True
__HAVE_DATA__ = False
# Reentrancy-Guard (verhindert Doppel-Starts)
__RUNNING__ = False

# Ampel-Schwellen (anpassbar)
THRESHOLDS = {
    # Kern
    "Cosine (TF-IDF)": (0.60, 0.75),
    "SBERT Cosine":    (0.75, 0.85),
    "BERTScore F1":    (0.85, 0.90),
    "ROUGE-L F1":      (0.45, 0.60),
    "BLEU-4":          (0.15, 0.25),

    # ROUGE-Details
    "ROUGE-L P": (0.45, 0.60), "ROUGE-L R": (0.45, 0.60),
    "ROUGE-2 P": (0.20, 0.35), "ROUGE-2 R": (0.20, 0.35),
    "ROUGE-1 P": (0.40, 0.60), "ROUGE-1 R": (0.40, 0.60),

    # RAGAS (0..1)
    "RAGAS: semantic_similarity": (0.70, 0.85),
    "RAGAS: answer_similarity":   (0.70, 0.85),
    "RAGAS: faithfulness":        (0.60, 0.75),
    "RAGAS: answer_relevancy":    (0.60, 0.75),
    "RAGAS: coverage":            (0.50, 0.70),
    "RAGAS: conciseness":         (0.60, 0.80),
    "RAGAS: fluency(FRE)":        (0.60, 0.75),
}

# --- Anzeige-Mapping für RAGAS-Header (nur UI, Daten bleiben gleich) ---
RAGAS_DISPLAY_MAP = {
    "RAGAS: semantic_similarity": "RAGAS:\nsemantic\nsimilarity",
    "RAGAS: answer_similarity":   "RAGAS:\nanswer\nsimilarity",
    "RAGAS: faithfulness":        "RAGAS:\nfaithfulness",
    "RAGAS: answer_relevancy":    "RAGAS:\nanswer\nrelevancy",
    "RAGAS: coverage":            "RAGAS:\ncoverage",
    "RAGAS: conciseness":         "RAGAS:\nconciseness",
    "RAGAS: fluency(FRE)":        "RAGAS:\nfluency\n(FRE)",
}
DISPLAY_TO_BASE = {v: k for k, v in RAGAS_DISPLAY_MAP.items()}

def _norm_col_key(k: str) -> str:
    """Normalisiert Spaltennamen (Dashes, Whitespaces) und mappt Anzeigeheader -> Basisnamen."""
    if k is None:
        return ""
    k = str(k)
    # geschützte / en dash / em dash vereinheitlichen
    k = k.replace("\u2011", "-").replace("\u2013", "-").replace("\u2014", "-")
    k = re.sub(r"\s+", " ", k).strip()
    # Anzeige (mit \n) -> Basisname
    return DISPLAY_TO_BASE.get(k, k)

# Farben
COLOR_OK   = "#d9f2d9"
COLOR_WARN = "#fff2b3"
COLOR_BAD  = "#ffd6d6"

try:    LEN_RATIO_BAND
except: LEN_RATIO_BAND = (0.85, 1.15)
try:    SYL_RATIO_BAND
except: SYL_RATIO_BAND = (0.85, 1.15)

def _color_for_kpi(col, v):
    """Ampelfarbe für KPI; robust gegen Anzeigeheader & Strings."""
    try:
        v = float(v)
    except Exception:
        return ""
    if pd.isna(v):
        return ""

    base_col = _norm_col_key(col)

    # dynamische Bänder
    if base_col == "len_ratio":
        low, high = LEN_RATIO_BAND
        if low <= v <= high: return COLOR_OK
        if (low - 0.15) <= v <= (high + 0.15): return COLOR_WARN
        return COLOR_BAD

    if base_col == "syl_ratio":
        low, high = SYL_RATIO_BAND
        if low <= v <= high: return COLOR_OK
        if (low - 0.15) <= v <= (high + 0.15): return COLOR_WARN
        return COLOR_BAD

    # regulär
    thr = THRESHOLDS.get(base_col)
    if thr is None:
        # Fallback für 0..1-Scores (wenn kein THRESHOLD hinterlegt)
        if 0.0 <= v <= 1.0:
            if v >= 0.75: return COLOR_OK
            if v >= 0.50: return COLOR_WARN
            return COLOR_BAD
        return ""

    y, g = thr
    if v >= g: return COLOR_OK
    if v >= y: return COLOR_WARN
    return COLOR_BAD

def _scroll_top():
    """Scrollt die Notebook-Ansicht sanft nach oben."""
    try:
        display(Javascript(
            "window.scrollTo({top: 0, behavior: 'smooth'});"
        ))
    except Exception:
        # Fallback (sollte praktisch nie gebraucht werden)
        display(HTML("<script>window.scrollTo({top:0,behavior:'smooth'});</script>"))

# =======================
# 📂 MDR & KI Loader – EIN Loader (Upload ODER Pfad), ersetzt Beispieltexte
# =======================

# (A) robuste Zeitparser
def _parse_time_to_seconds(t: str):
    if t is None: return np.nan
    ts = str(t).strip().replace("\u2009","").replace("\u00a0","")
    parts = [p for p in ts.split(":") if p]
    if len(parts) == 2 and parts[0].isdigit() and int(parts[0]) > 59: parts = parts[1:]
    if len(parts) >= 2 and parts[0] == "10": parts = parts[1:]
    try:
        if len(parts)==1: return float(parts[0])
        if len(parts)==2: m,s=parts; return int(m)*60+float(s)
        if len(parts)==3: h,m,s=parts; return int(h)*3600+int(m)*60+float(s)
        nums = re.findall(r"\d+", ts)
        return (int(nums[-2])*60+int(nums[-1])) if len(nums)>=2 else np.nan
    except: 
        return np.nan

def _fmt_seconds(sec: float):
    if pd.isna(sec): return ""
    sec=int(round(float(sec)))
    if sec<3600: m,s=divmod(sec,60); return f"{m:02d}:{s:02d}"
    h,r=divmod(sec,3600); m,s=divmod(r,60); return f"{h:d}:{m:02d}:{s:02d}"

# (B) Loader für TXT: 3 Spalten (start\tende\ttext) ODER 2 Spalten (time\ttext)
def load_ad_txt(path: str) -> pd.DataFrame:
    df = None
    # 1) Versuche TSV
    for header in [0, None]:
        try:
            tmp = pd.read_csv(path, sep="\t", header=header, dtype=str, engine="python")
            if tmp.shape[1] >= 3:
                tmp = tmp.iloc[:, :3]
                tmp.columns = ["start", "ende", "text"]
                df = tmp
                break
            if tmp.shape[1] == 2:
                tmp.columns = ["start", "text"]
                tmp["ende"] = ""                     # <— WICHTIG
                tmp = tmp[["start", "ende", "text"]] # konsistente Reihenfolge
                df = tmp
                break
        except Exception:
            pass

    # 2) Fallback: freies Parsen (1–3 Felder pro Zeile)
    if df is None:
        rows = []
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                line = line.rstrip("\n")
                parts = line.split("\t") if "\t" in line else re.split(r"\s{2,}", line)
                parts = [p.strip() for p in parts if p.strip()]
                if len(parts) >= 3:
                    rows.append(parts[:3])
                elif len(parts) == 2:
                    rows.append([parts[0], "", parts[1]])
                elif len(parts) == 1:
                    rows.append(["", "", parts[0]])
        df = pd.DataFrame(rows, columns=["start", "ende", "text"])

    # 3) Säubern (nur vorhandene Spalten)
    for c in [c for c in ["start", "ende", "text"] if c in df.columns]:
        df[c] = (
            df[c].fillna("")
                 .astype(str)
                 .str.replace(r"\s+", " ", regex=True)
                 .str.strip()
        )

    # 4) Zeiten berechnen
    df["t_start"] = df["start"].map(_parse_time_to_seconds)
    has_end = df["ende"].replace("", np.nan).notna().any()
    df["t_end"]   = df["ende"].map(_parse_time_to_seconds) if has_end else np.nan

    # Falls nur Startzeiten: Ende = nächster Start bzw. +2s
    mask = df["t_start"].notna()
    t = df.loc[mask, "t_start"].to_list()
    df.loc[mask, "t_end"] = [t[i+1] if i < len(t)-1 else t[i] + 2.0 for i in range(len(t))]

    # formatierte Zeiten
    df["start"] = df["t_start"].map(_fmt_seconds)
    df["ende"]  = df["t_end"].map(_fmt_seconds)
    return df[["start", "ende", "text", "t_start", "t_end"]]

# =======================
# (C) UI – Upload-only + Buttons (Laden / Berechnen / Reset) – bündige Breiten
# =======================

# --- Breiten/Layouts
BTN_W   = "220px"
DESC_W  = "92px"
PROG_W  = "100%"
UPLOAD_W = BTN_W

u_mdr = W.FileUpload(accept=".txt", multiple=False, layout=W.Layout(width=UPLOAD_W))
u_ki  = W.FileUpload(accept=".txt", multiple=False, layout=W.Layout(width=UPLOAD_W))

btn_load  = W.Button(description="Laden",     icon="upload", button_style="primary",
                     layout=W.Layout(width=BTN_W, height="36px"))
btn_calc  = W.Button(description="Berechnen", icon="gear",   button_style="primary",
                     layout=W.Layout(width=BTN_W, height="36px"))
btn_reset = W.Button(description="Reset",     icon="trash",  button_style="",
                     layout=W.Layout(width="140px", height="36px"))

# Fortschritt
p_load = W.IntProgress(description="Laden:", min=0, max=100, value=0, bar_style="info",
                       layout=W.Layout(width=PROG_W), style={"description_width": DESC_W})
p_calc = W.IntProgress(description="Berechnen:", min=0, max=100, value=0, bar_style="info",
                       layout=W.Layout(width=PROG_W), style={"description_width": DESC_W})

# >>> Overflow-Settings für Widgets ...
for _w in (u_mdr, u_ki, btn_load, btn_calc, btn_reset, p_load, p_calc):
    _w.layout.overflow = 'visible'

def _progress_calc(frac: float):
    """Setzt den Berechnungsfortschritt (0..1 → 0..100)."""
    try:
        frac = float(frac)
    except Exception:
        frac = 0.0
    p_calc.value = max(0, min(100, int(round(frac * 100))))
    p_calc.bar_style = "success" if p_calc.value >= 100 else "info"

progress_card = W.VBox(
    [W.HTML("<div style='font-weight:600;margin-bottom:6px'>Fortschrittsanzeige</div>"), p_load, p_calc],
    layout=W.Layout(padding="10px 12px", border="1px solid #e5e7eb", border_radius="8px",
                    margin="6px 0 0 0", width="100%")
)

msg = W.HTML()
btn_calc.disabled = True

def _save_upload(widget: W.FileUpload, target_path: str) -> str | None:
    v = widget.value
    if not v: return None
    meta = next(iter(v.values())) if isinstance(v, dict) else v[0]
    content = meta["content"] if isinstance(meta, dict) else getattr(meta, "content", None)
    if content is None: raise ValueError("Upload-Widget ohne 'content'")
    with open(target_path, "wb") as f: f.write(content)
    try: widget.value = ()
    except Exception:
        try: widget.value = {}
        except Exception: pass
    return target_path

def _set_compute_enabled():
    have = isinstance(globals().get("ref_segs"), list) and len(globals().get("ref_segs", [])) \
        and isinstance(globals().get("hyp_segs"), list) and len(globals().get("hyp_segs", []))
    btn_calc.disabled = not bool(have)

def _unique_path(base: str, ext: str) -> str:
    ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    path = f"/mnt/data/{base}_{ts}{ext}"
    if not os.path.exists(path):
        return path
    i = 1
    while True:
        alt = f"/mnt/data/{base}_{ts}_{i}{ext}"
        if not os.path.exists(alt):
            return alt
        i += 1

def _export_txt_safely(df: pd.DataFrame) -> tuple[str, str | None]:
    """Speichert 1) UTF-8-SIG (Excel freundlich) und optional 2) UTF-16-TSV."""
    if df is None or df.empty:
        raise RuntimeError("Keine Ergebnisse – bitte zuerst Berechnen.")
    out_txt = _unique_path("ad_compare_export", ".txt")
    # numerische Spalten runden (nur floats)
    num_cols = [c for c in df.columns if pd.api.types.is_float_dtype(df[c])]
    if num_cols:
        df[num_cols] = df[num_cols].round(4)
    # 1) UTF-8 mit BOM → Excel liest sauber
    df.to_csv(out_txt, sep="\t", index=False, encoding="utf-8-sig")

    # 2) optional: UTF-16 (manche Excel-Builds mögen das)
    out_tsv16 = None
    try:
        out_tsv16 = _unique_path("ad_compare_export_excel", ".tsv")
        df.to_csv(out_tsv16, sep="\t", index=False, encoding="utf-16", errors="ignore")
    except Exception:
        out_tsv16 = None
    return out_txt, out_tsv16

# ---- Tabs-Output FRÜH anlegen (vor jeder Verwendung!) ----
tabs_out = W.Output()

charts_out = W.Output()
summary_out = W.Output()

with charts_out:
    charts_out.clear_output()
    display(HTML("<em>Noch keine Charts – bitte zuerst <b>Berechnen</b>.</em>"))
with summary_out:
    summary_out.clear_output()
    display(HTML("<em>Noch keine Kennzahlen – bitte zuerst <b>Berechnen</b>.</em>"))
    
tabs_out.layout.max_height = 'none'
tabs_out.layout.overflow_y = 'visible'
tabs_out.layout.overflow_x = 'hidden'

with tabs_out:
    tabs_out.clear_output()
    display(HTML("<em>Noch keine Daten. Bitte MDR & KI laden und dann auf <b>Berechnen</b> klicken.</em>"))

def _reset_state():
    for k in ["ref_segs","hyp_segs","pairs","df","table","ragas_seg","ref_tok","hyp_tok","S","report"]:
        globals().pop(k, None)
    globals()["__HAVE_DATA__"] = False
    p_load.value = 0; p_load.bar_style = "info"
    p_calc.value = 0; p_calc.bar_style = "info"
    msg.value = "<div style='color:#4b5563'>Zurückgesetzt. Bitte neue Dateien laden.</div>"
    _set_compute_enabled()
    with tabs_out:
        tabs_out.clear_output()
        display(HTML("<em>Noch keine Daten. Bitte MDR & KI laden und dann auf <b>Berechnen</b> klicken.</em>"))
    with charts_out:
        charts_out.clear_output()
        display(HTML("<em>Noch keine Charts – bitte zuerst <b>Berechnen</b>.</em>"))

header = W.HTML(
    '<div style="padding:10px 12px;border:1px solid #e5e7eb;border-radius:12px;background:#fafafa">'
    '<div style="font-weight:600;margin-bottom:6px">Daten laden (MDR & KI)</div>'
    '<ol style="color:#4b5563;font-size:12px;margin:0 0 4px 18px;padding:0;line-height:1.4">'
    '<li>Beide <b>.txt</b>-Dateien über die Upload-Buttons auswählen.</li>'
    '<li>Auf <b>Laden</b> klicken.</li>'
    '<li>Mit <b>Berechnen</b> die Auswertung starten.</li>'
    '<li><b>Reset</b> setzt alles zurück für einen neuen Vergleich.</li>'
    '</ol>'
    '</div>'
)

# Upload-Spalten + Grid
col_mdr = W.VBox([W.HTML("<b>MDR .txt</b>"), u_mdr], layout=W.Layout(width=BTN_W))
col_ki  = W.VBox([W.HTML("<b>KI .txt</b>"),  u_ki ], layout=W.Layout(width=BTN_W))
spacer_top = W.Box(layout=W.Layout(width=BTN_W, height="32px"))

grid_panel = W.GridBox(
    children=[col_mdr, col_ki, spacer_top, btn_load, btn_calc, btn_reset],
    layout=W.Layout(
        grid_template_columns=f"{BTN_W} {BTN_W} {BTN_W}",
        grid_auto_rows="min-content",
        grid_gap="10px",
        align_items="center",
        justify_items="flex-start",  # <— statt "start"
    )
)

# >>> NEU: Container sollen selbst nicht horizontal scrollen
col_mdr.layout.overflow = 'visible'
col_ki.layout.overflow  = 'visible'
grid_panel.layout.overflow = 'visible'
progress_card.layout.overflow = 'visible'

# Obercontainer
box = W.VBox(
    [header, grid_panel, progress_card, tabs_out, msg],
    layout=W.Layout(gap="10px", width="100%", overflow='visible', overflow_x='hidden')
)

display(box)

def on_load_click(_):
    globals()["__RUNNING__"] = False

    try:
        msg.value = ""
        p_load.value = 5;  p_load.bar_style = "info"
        for k in ["pairs","df","table","ragas_seg","ref_tok","hyp_tok","S","report"]:
            globals().pop(k, None)
        globals()["__HAVE_DATA__"] = False
        _set_compute_enabled()

        mdr_saved = _save_upload(u_mdr, "/mnt/data/_mdr.txt")
        ki_saved  = _save_upload(u_ki,  "/mnt/data/_ki.txt")
        if not mdr_saved or not ki_saved:
            msg.value = "<div style='color:#b00020'>Bitte beide Dateien auswählen und erneut auf <b>Laden</b> klicken.</div>"
            p_load.value = 0; p_load.bar_style = "danger"
            return

        p_load.value = 30
        mdr_df = load_ad_txt(mdr_saved);  p_load.value = 55
        ki_df  = load_ad_txt(ki_saved);   p_load.value = 75

        # ⬇️ neu: für spätere Exporte/Zeitspalten aufbewahren
        globals()["mdr_df"] = mdr_df.copy()
        globals()["ki_df"]  = ki_df.copy()

        globals()["ref_segs"] = [s for s in mdr_df["text"].tolist() if s.strip()]
        globals()["hyp_segs"] = [s for s in ki_df["text"].tolist()  if s.strip()]
        p_load.value = 100; p_load.bar_style = "success"
        msg.value = (f"<div style='color:#2e7d32'>OK – {len(ref_segs)} Referenz- und {len(hyp_segs)} KI-Segmente geladen.</div>")
        _scroll_top()
    except Exception as e:
        p_load.bar_style = "danger"
        msg.value = f"<div style='color:#b00020'>Fehler beim Laden: {e}</div>"
    finally:
        _set_compute_enabled()

btn_load.on_click(on_load_click)
btn_reset.on_click(lambda _: _reset_state())

# =======================
# 🔌 Optionale Bibliotheken (robust)
# =======================
_HAS_ST = CONFIG["USE_SBERT"]
_HAS_BERTSCORE = CONFIG["USE_BERTSCORE"]
_HAS_SPACY = CONFIG["USE_SPACY"]
_HAS_RAGAS = CONFIG["USE_RAGAS"]

try:
    from sentence_transformers import SentenceTransformer
except Exception:
    _HAS_ST = False

try:
    from bert_score import score as bertscore_score
except Exception:
    _HAS_BERTSCORE = False

_nlp_de = None
if _HAS_SPACY:
    try:
        import spacy
        try:
            _nlp_de = spacy.load("de_core_news_sm", disable=["ner","parser","textcat"])
        except Exception:
            _HAS_SPACY = False
    except Exception:
        _HAS_SPACY = False

# RAGAS (optional, defensiv)
_ragas_available = False
_ragas_metrics = {}
if _HAS_RAGAS:
    try:
        import ragas  # noqa
        try:
            from ragas.metrics import (
                faithfulness as rg_faithfulness,
                answer_relevancy as rg_answer_relevancy,
            )
            _ragas_metrics["faithfulness"] = rg_faithfulness
            _ragas_metrics["answer_relevancy"] = rg_answer_relevancy
        except Exception:
            pass
        try:
            from ragas.metrics import semantic_similarity as rg_semantic_similarity
            _ragas_metrics["semantic_similarity"] = rg_semantic_similarity
        except Exception:
            pass
        try:
            from ragas.metrics import answer_similarity as rg_answer_similarity
            _ragas_metrics["answer_similarity"] = rg_answer_similarity
        except Exception:
            pass
        try:
            from ragas.metrics import coverage as rg_coverage
            _ragas_metrics["coverage"] = rg_coverage
        except Exception:
            pass
        try:
            from ragas.metrics import conciseness as rg_conciseness
            _ragas_metrics["conciseness"] = rg_conciseness
        except Exception:
            pass
        _ragas_available = len(_ragas_metrics) > 0
    except Exception:
        _ragas_available = False

# =======================
# 🎨 Domänenspezifische Erkennung (Farben & Bewegung)
# =======================
COLOR_STEMS = [
    "schwarz","weiß","weiss","grau","rot","blau","grün","gruen","gelb","orange",
    "violett","lila","braun","beige","gold","silber","türkis","tuerkis","magenta","rosa"
]
COLOR_PREFIXES = ["hell","dunkel","knall","neon","pastell"]

def _strip_color_affixes(piece: str) -> str:
    x = piece
    x = re.sub(r"farb\w+$", "", x)
    for pref in COLOR_PREFIXES:
        if x.startswith(pref):
            x = x[len(pref):]
            break
    return x

def has_color_tokens(tokens: list[str]) -> bool:
    for tok in (t.lower() for t in tokens):
        for part in tok.split('-'):
            base = _strip_color_affixes(part)
            for stem in COLOR_STEMS:
                if base.startswith(stem):
                    if stem == "rosa" and not re.match(r"rosa($|[a-zäöüß]*?(?:e|en|em|er|es|farb))", base):
                        continue
                    return True
    return False

COLOR_LEMMAS = set(COLOR_STEMS)
MOTION_LEMMAS = {
    "gehen","kommen","laufen","rennen","springen","schreiten",
    "drehen","heben","senken","zeigen","blicken","schauen",
    "fahren","wenden","betreten","treten","setzen","nähern","bewegen"
}
MOTION_MULTI = {"stehen bleiben","sich setzen","sich drehen","sich nähern","aufstehen","hinsetzen"}

def word_tokenize(text: str) -> List[str]:
    text = re.sub(r"\s+"," ", text.lower()).strip()
    return re.findall(r"[A-Za-zÄÖÜäöüß0-9\-]+", text)

def lemmas_de(text: str) -> list[str]:
    if _HAS_SPACY and _nlp_de is not None:
        doc = _nlp_de(text)
        return [t.lemma_.lower() for t in doc if not t.is_punct and not t.is_space]
    return [t.lower() for t in word_tokenize(text)]

def has_motion_text(text: str) -> bool:
    L = lemmas_de(text)
    lemma_str = " ".join(L)
    if any(p in lemma_str for p in MOTION_MULTI): return True
    return any(l in MOTION_LEMMAS for l in L)

def has_color_lemma(tokens: list[str]) -> bool:
    if not _HAS_SPACY: return has_color_tokens(tokens)
    doc = _nlp_de(" ".join(tokens))
    lemmas = {t.lemma_.lower() for t in doc}
    texts  = [t.text.lower() for t in doc]
    if lemmas & COLOR_LEMMAS: return True
    for t in texts:
        for part in t.split("-"):
            base = _strip_color_affixes(part)
            if base in COLOR_LEMMAS: return True
            if any(base.endswith(c) for c in COLOR_LEMMAS): return True
    return False

def has_motion_tokens(tokens: list[str]) -> bool:
    G = {
        "gehen","geht","ging","gegangen","kommen","kommt","kam","gekommen","laufen","läuft","lief","gelaufen","laeuft",
        "rennen","rennt","rannte","gerannt","springen","springt","sprang","gesprungen","drehen","dreht","drehte","gedreht",
        "heben","hebt","hob","gehoben","senken","senkt","senkte","gesenkt","zeigen","zeigt","zeigte","gezeigt",
        "blicken","blickt","blickte","geblickt","schauen","schaut","schaute","geschaut",
        "schreiten","schreitet","schritt","geschritten","fahren","fährt","fuhr","gefahren","faehrt","wenden","wendet","wandte","gewandt"
    }
    toks = {t.lower() for t in tokens}
    return any(t in G for t in toks)

# =======================
# 🔤 Satz- & Wort-Utilities
# =======================
def normalize_text(text: str) -> str:
    return re.sub(r"\s+"," ", text.lower()).strip()

def sent_tokenize(text: str) -> List[str]:
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if len(lines) >= 2 and (sum(len(l) for l in lines)/max(len(lines),1)) > 10:
        return lines
    parts = re.split(r"(?<=[\.\!\?])\s+", text)
    return [p.strip() for p in parts if p.strip()]

def ngrams(tokens: List[str], n: int):
    return [tuple(tokens[i:i+n]) for i in range(0, len(tokens)-n+1)]

# =======================
# 🗣️ BLEU / ROUGE
# =======================
def bleu_score(reference: List[str], hypothesis: List[str], max_n: int = 4) -> float:
    precisions = []
    for n in range(1, max_n+1):
        ref_ngrams = Counter(ngrams(reference, n))
        hyp_ngrams = Counter(ngrams(hypothesis, n))
        total = sum(hyp_ngrams.values())
        if total == 0: precisions.append(0.0); continue
        overlap = sum(min(cnt, ref_ngrams.get(ng,0)) for ng, cnt in hyp_ngrams.items())
        precisions.append(overlap / total if total > 0 else 0.0)
    ref_len, hyp_len = len(reference), len(hypothesis)
    if hyp_len == 0: return 0.0
    bp = 1.0 if hyp_len > ref_len else math.exp(1 - ref_len / max(hyp_len, 1))
    if any(p == 0 for p in precisions): return 0.0
    log_prec = sum(math.log(p) for p in precisions) / len(precisions)
    return float(bp * math.exp(log_prec))

def rouge_n(reference: List[str], hypothesis: List[str], n: int = 1) -> Dict[str, float]:
    ref_ngrams = Counter(ngrams(reference, n))
    hyp_ngrams = Counter(ngrams(hypothesis, n))
    overlap = sum(min(cnt, hyp_ngrams.get(ng,0)) for ng, cnt in ref_ngrams.items())
    ref_total = sum(ref_ngrams.values()); hyp_total = sum(hyp_ngrams.values())
    recall = overlap / ref_total if ref_total > 0 else 0.0
    precision = overlap / hyp_total if hyp_total > 0 else 0.0
    f1 = (2*precision*recall)/(precision+recall) if (precision+recall) > 0 else 0.0
    return {"precision": float(precision), "recall": float(recall), "f1": float(f1)}

def lcs_length(a: List[str], b: List[str]) -> int:
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            dp[i][j] = dp[i-1][j-1] + 1 if a[i-1] == b[j-1] else max(dp[i-1][j], dp[i][j-1])
    return dp[m][n]

def rouge_l(reference: List[str], hypothesis: List[str]) -> Dict[str, float]:
    lcs = lcs_length(reference, hypothesis)
    ref_len, hyp_len = len(reference), len(hypothesis)
    recall = lcs / ref_len if ref_len > 0 else 0.0
    precision = lcs / hyp_len if hyp_len > 0 else 0.0
    f1 = (2*precision*recall)/(precision+recall) if (precision+recall) > 0 else 0.0
    return {"precision": float(precision), "recall": float(recall), "f1": float(f1)}

# =======================
# 🔎 TF-IDF & Cosine
# =======================
def build_tfidf_vectors(docs_tokens: List[List[str]]):
    vocab = sorted(set(t for doc in docs_tokens for t in doc))
    term_index = {t:i for i,t in enumerate(vocab)}
    tf = np.zeros((len(docs_tokens), len(vocab)), dtype=float)
    for di, toks in enumerate(docs_tokens):
        counts = Counter(toks); total = sum(counts.values()) or 1
        for t, c in counts.items():
            tf[di, term_index[t]] = c / total
    df = np.array([sum(1 for doc in docs_tokens if t in set(doc)) for t in vocab], dtype=float)
    idf = np.log((1 + len(docs_tokens)) / (1 + df)) + 1.0
    tfidf = tf * idf
    return tfidf, vocab

def cosine_similarity_matrix(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    A_norm = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-12)
    B_norm = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-12)
    return A_norm @ B_norm.T

def greedy_alignment(ref_segments_tokens: List[List[str]], hyp_segments_tokens: List[List[str]]):
    all_docs = ref_segments_tokens + hyp_segments_tokens
    tfidf, _ = build_tfidf_vectors(all_docs)
    n_ref = len(ref_segments_tokens)
    A, B = tfidf[:n_ref, :], tfidf[n_ref:, :]
    S = cosine_similarity_matrix(A, B)
    used_hyp = set(); pairs = []
    for r in range(n_ref):
        order = np.argsort(-S[r]); h_choice = None
        for h in order:
            if h not in used_hyp: h_choice = h; break
        if h_choice is None: h_choice = int(order[0])
        pairs.append((r, h_choice, float(S[r, h_choice]))); used_hyp.add(h_choice)
    return pairs, S

# =======================
# 🔢 Silben & Lesbarkeit (DE, Flesch–Amstad)
# =======================
_VOWELS_DE = set("aeiouyäöü")
def count_syllables_de_word(w: str) -> int:
    w = w.lower()
    w = re.sub(r"[^a-zäöüß]", "", w)
    if not w: return 0
    w2 = re.sub(r"ie", "i", w)
    w2 = re.sub(r"aa|ee|oo", lambda m: m.group(0)[0], w2)
    groups = re.findall(r"[aeiouyäöü]+", w2)
    syl = len(groups)
    if w.endswith("e"): syl += 0
    return max(syl, 1)

def count_syllables_text(text: str) -> int:
    toks = word_tokenize(text)
    return sum(count_syllables_de_word(t) for t in toks)

def flesch_amstad(text: str) -> float:
    sentences = sent_tokenize(text)
    n_sent = max(len(sentences), 1)
    toks = word_tokenize(text)
    n_words = max(len(toks), 1)
    n_syl = sum(count_syllables_de_word(t) for t in toks)
    ASL = n_words / n_sent
    ASW = n_syl / n_words
    fre = 180 - ASL - (58.5 * ASW)
    return float(np.clip(fre, -20, 130))

# =======================
# 📏 Segmentmetriken (inkl. Lemma-Checks & Silben)
# =======================
def contains_any(tokens: List[str], vocab: set[str]) -> bool:
    return any(t in vocab for t in tokens)

def segment_metrics(ref_tokens: List[str], hyp_tokens: List[str]) -> Dict[str, Any]:
    res = {}
    res["bleu_1"] = bleu_score(ref_tokens, hyp_tokens, max_n=1)
    res["bleu_2"] = bleu_score(ref_tokens, hyp_tokens, max_n=2)
    res["bleu_3"] = bleu_score(ref_tokens, hyp_tokens, max_n=3)
    res["bleu_4"] = bleu_score(ref_tokens, hyp_tokens, max_n=4)
    r1 = rouge_n(ref_tokens, hyp_tokens, n=1)
    r2 = rouge_n(ref_tokens, hyp_tokens, n=2)
    rl = rouge_l(ref_tokens, hyp_tokens)
    for k,v in r1.items(): res[f"rouge1_{k}"] = v
    for k,v in r2.items(): res[f"rouge2_{k}"] = v
    for k,v in rl.items(): res[f"rougeL_{k}"] = v

    # Heuristiken
    res["ref_has_color"]  = has_color_lemma(ref_tokens)
    res["hyp_has_color"]  = has_color_lemma(hyp_tokens)
    ref_txt = " ".join(ref_tokens)
    hyp_txt = " ".join(hyp_tokens)
    res["ref_has_motion"] = has_motion_text(ref_txt) if _HAS_SPACY else has_motion_tokens(ref_tokens)
    res["hyp_has_motion"] = has_motion_text(hyp_txt) if _HAS_SPACY else has_motion_tokens(hyp_tokens)

    # Längen/Silben
    res["len_ref"] = len(ref_tokens); res["len_hyp"] = len(hyp_tokens)
    res["syl_ref"] = sum(count_syllables_de_word(t) for t in ref_tokens)
    res["syl_hyp"] = sum(count_syllables_de_word(t) for t in hyp_tokens)
    return res

# =======================
# 🤖 SBERT & BERTScore (optional)
# =======================
_ST_MODEL = None
def get_st_model(model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
    global _ST_MODEL
    if not _HAS_ST: return None
    if _ST_MODEL is None:
        _ST_MODEL = SentenceTransformer(model_name)
    return _ST_MODEL

def sentence_embeddings(sentences: List[str]):
    if not _HAS_ST: return None
    model = get_st_model()
    try:
        return model.encode(sentences, convert_to_numpy=True, normalize_embeddings=True)
    except Exception:
        return None

def bertscore_pairs(ref_segments, hyp_segments, model_type: str = "bert-base-multilingual-cased"):
    if not _HAS_BERTSCORE: return None
    try:
        f1 = []
        for r, h in zip(ref_segments, hyp_segments):
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                P, R, F1 = bertscore_score([h], [r], model_type=model_type, verbose=False)
            f1.append(float(F1.mean().item()))
        return f1
    except Exception:
        return None

# =======================
# 🧰 Stopwörter (für Coverage-Fallback)
# =======================
GER_STOP = {
    "der","die","das","und","oder","ein","eine","einer","einem","einen","den","dem","des",
    "zu","auf","im","in","am","an","ist","war","sind","sein","mit","von","für","als","auch",
    "sich","sie","er","es","wir","ihr","ihn","ihm","man","dass","so","wie","nicht","nur","noch","schon"
}

def content_tokens(tokens: List[str]) -> List[str]:
    return [t for t in tokens if t not in GER_STOP and not t.isdigit()]

# =======================
# 🧪 RAGAS (optional) – Fallback-Metriken pro Segment
# =======================
def ragas_like_metrics(ref_segs: List[str], hyp_segs: List[str], df_base: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for i, (r, h) in enumerate(zip(ref_segs, hyp_segs)):
        rt = word_tokenize(r); ht = word_tokenize(h)
        rL   = df_base.loc[i, "rougeL_f1"]        if "rougeL_f1"        in df_base.columns else np.nan
        rL_P = df_base.loc[i, "rougeL_precision"] if "rougeL_precision" in df_base.columns else np.nan
        rL_R = df_base.loc[i, "rougeL_recall"]    if "rougeL_recall"    in df_base.columns else np.nan
        sbert= df_base.loc[i, "sbert_cosine"]     if "sbert_cosine"     in df_base.columns else np.nan
        bsf1 = df_base.loc[i, "bertscore_f1"]     if "bertscore_f1"     in df_base.columns else np.nan

        def clamp01(x):
            try: return float(np.clip(x, 0, 1))
            except: return np.nan

        sem_sim = sbert if not pd.isna(sbert) else rL
        ans_sim = bsf1  if not pd.isna(bsf1)  else rL
        faith   = rL_R
        ans_rel = rL_P
        ct_r = set(content_tokens(rt)); ct_h = set(content_tokens(ht))
        cov = len(ct_r & ct_h) / (len(ct_r) or 1)
        lr = (len(ht) / (len(rt) or 1))
        conc = 1 - min(1.0, abs(lr - 1.0))
        fre_seg = flesch_amstad(h)
        flu = float(np.clip(fre_seg/100.0, 0, 1))

        rows.append({
            "RAGAS: semantic_similarity": clamp01(sem_sim),
            "RAGAS: answer_similarity":   clamp01(ans_sim),
            "RAGAS: faithfulness":        clamp01(faith),
            "RAGAS: answer_relevancy":    clamp01(ans_rel),
            "RAGAS: coverage":            clamp01(cov),
            "RAGAS: conciseness":         clamp01(conc),
            "RAGAS: fluency(FRE)":        clamp01(flu),
        })
    return pd.DataFrame(rows)     

# --- Reentrancy-Flag oben bleibt ---
__RUNNING__ = False

# =======================
# 🚀 Pipeline ausführen – Button-gesteuert (kein Fallback, kein ValueError)
# =======================
def compute_pipeline(_=None):
    global df, table, ragas_seg, ref_tok, hyp_tok, pairs, S, __RUNNING__, __HAVE_DATA__

    # 1) Daten vorhanden?
    if not (isinstance(globals().get("ref_segs"), list) and len(ref_segs)
            and isinstance(globals().get("hyp_segs"), list) and len(hyp_segs)):
        msg.value = "<div style='color:#b00020'>Bitte oben beide Dateien laden und dann <b>Berechnen</b> klicken.</div>"
        return

    _scroll_top()

    # 2) Reentrancy
    if __RUNNING__:
        return
    __RUNNING__ = True
    btn_calc.disabled = True

    try:
        _progress_calc(0.05)

        # 3) Tokenisieren & Alignment
        ref_tok = [word_tokenize(s) for s in ref_segs]
        hyp_tok = [word_tokenize(s) for s in hyp_segs]
        pairs, S = greedy_alignment(ref_tok, hyp_tok)
        _progress_calc(0.25)

        # 4) Segmentmetriken
        rows = []
        for r_idx, h_idx, sim in pairs:
            m = segment_metrics(ref_tok[r_idx], hyp_tok[h_idx])
            row = {"ref_idx": r_idx, "hyp_idx": h_idx, "alignment_cosine": sim,
                   "ref_segment": ref_segs[r_idx], "hyp_segment": hyp_segs[h_idx]}
            row.update(m)
            rows.append(row)
        df = pd.DataFrame(rows)
        _progress_calc(0.55)

        # 5) Optional: SBERT/BERTScore
        emb_ref = sentence_embeddings(df["ref_segment"].tolist()) if (_HAS_ST and not df.empty and __COMPUTE_SBERT__) else None
        emb_hyp = sentence_embeddings(df["hyp_segment"].tolist()) if (_HAS_ST and not df.empty and __COMPUTE_SBERT__) else None
        df["sbert_cosine"] = (emb_ref * emb_hyp).sum(axis=1) if (emb_ref is not None and emb_hyp is not None) else np.nan

        bs = bertscore_pairs(df["ref_segment"].tolist(), df["hyp_segment"].tolist()) if (_HAS_BERTSCORE and not df.empty and __COMPUTE_BERTSCORE__) else None
        df["bertscore_f1"] = bs if bs is not None else np.nan
        _progress_calc(0.75)

        # 6) RAGAS-Fallbacks + Tabelle
        ragas_seg = ragas_like_metrics(df["ref_segment"].tolist(), df["hyp_segment"].tolist(), df)
        _progress_calc(0.90)
    
        front = [c for c in ["ref_segment","hyp_segment"] if c in df.columns]
        pref  = [
            "alignment_cosine","sbert_cosine","bertscore_f1",
            "rougeL_f1","rouge2_f1","rouge1_f1",
            "bleu_4","bleu_3","bleu_2","bleu_1",
            "rougeL_precision","rougeL_recall","rouge2_precision","rouge2_recall","rouge1_precision","rouge1_recall",
            "len_ref","len_hyp","syl_ref","syl_hyp","ref_has_color","hyp_has_color","ref_has_motion","hyp_has_motion",
            "ref_idx","hyp_idx"
        ]

        # ⬇️ neu: Zeitspalten aus den Original-Tabellen anhand der Indizes mappen
        _mdr = globals().get("mdr_df", None)
        _ki  = globals().get("ki_df",  None)

        if _mdr is not None and _ki is not None:
            def _safe_get(df_, i, col):
                try:
                    return df_.iloc[int(i)][col] if 0 <= int(i) < len(df_) else ""
                except Exception:
                    return ""
        
            df["start_ref"] = df["ref_idx"].map(lambda i: _safe_get(_mdr, i, "start"))
            df["ende_ref"]  = df["ref_idx"].map(lambda i: _safe_get(_mdr, i, "ende"))
            df["start_hyp"] = df["hyp_idx"].map(lambda i: _safe_get(_ki,  i, "start"))
            df["ende_hyp"]  = df["hyp_idx"].map(lambda i: _safe_get(_ki,  i, "ende"))

        # ⬇️ neu: Zeitspalten in die Front-Spalten aufnehmen
        front = [c for c in ["ref_segment","hyp_segment"] if c in df.columns]
        time_cols = ["start_ref","ende_ref","start_hyp","ende_hyp"]
        front = time_cols + front  # Zeit vor die Texte
        
        cols = front + [c for c in pref if c in df.columns]
        table = df[cols].rename(columns={
            "ref_segment":"Referenzsegment (MDR)",
            "hyp_segment":"Hypothesensegment (KI)",
            "alignment_cosine":"Cosine (TF-IDF)",
            "sbert_cosine":"SBERT Cosine",
            "bertscore_f1":"BERTScore F1",
            "rouge1_precision":"ROUGE-1 P","rouge1_recall":"ROUGE-1 R","rouge1_f1":"ROUGE-1 F1",
            "rouge2_precision":"ROUGE-2 P","rouge2_recall":"ROUGE-2 R","rouge2_f1":"ROUGE-2 F1",
            "rougeL_precision":"ROUGE-L P","rougeL_recall":"ROUGE-L R","rougeL_f1":"ROUGE-L F1",
            "bleu_1":"BLEU-1","bleu_2":"BLEU-2","bleu_3":"BLEU-3","bleu_4":"BLEU-4",
            "len_ref":"Wörter Ref","len_hyp":"Wörter Hyp",
            "syl_ref":"Silben Ref","syl_hyp":"Silben Hyp",
            "ref_has_color":"Ref Farbdetail","hyp_has_color":"Hyp Farbdetail",
            "ref_has_motion":"Ref Bewegung","hyp_has_motion":"Hyp Bewegung",
            "ref_idx":"Ref-Index","hyp_idx":"Hyp-Index"
        })
        table["len_ratio"] = np.where(table["Wörter Ref"]>0, table["Wörter Hyp"]/table["Wörter Ref"], np.nan)
        table["syl_ratio"] = np.where(table["Silben Ref"]>0, table["Silben Hyp"]/table["Silben Ref"], np.nan)
        _progress_calc(0.90)

        # 7) UI
        rebuild_tabs()
        __HAVE_DATA__ = True
        render_summary_and_report()
        # ⬇️ Charts & Kennzahlen gezielt in die Output-Container schreiben
        with charts_out:
            charts_out.clear_output(wait=True)
            draw_charts()

        with summary_out:
            summary_out.clear_output(wait=True)
            render_summary_and_report()

        msg.value = "<div style='color:#2e7d32'>Berechnung abgeschlossen und Tabs aktualisiert.</div>"
        _progress_calc(1.0)
        
    except Exception as e:
        msg.value = f"<div style='color:#b00020'>Fehler in der Berechnung: {escape(str(e))}</div>"
        p_calc.bar_style = "danger"
    finally:
        __RUNNING__ = False
        _set_compute_enabled()   # aktiviert Button wieder, falls Daten da

# ---- Button-Bindung (einmalig, außerhalb der Funktion!) ----
try:
    btn_calc._click_handlers.callbacks.clear()
except Exception:
    pass
btn_calc.on_click(compute_pipeline)

# =======================
# ⚙️ Settings-Panel (Akkordeon) + Rebuild-Mechanik
# =======================
UI_TEXT_WIDTH = 300
UI_NUM_MIN = 85
UI_NUM_MAX = 100

def build_settings_panel():
    # Verfügbarkeiten
    lib_row = W.HBox([
        W.HTML(f"<b>spaCy:</b> {'✅' if _HAS_SPACY else '❌'}"),
        W.HTML(f"<b>SBERT:</b> {'✅' if _HAS_ST else '❌'}"),
        W.HTML(f"<b>BERTScore:</b> {'✅' if _HAS_BERTSCORE else '❌'}"),
        W.HTML(f"<b>RAGAS:</b> {'✅' if _ragas_available else '❌'}")
    ], layout=W.Layout(justify_content="space-between"))

    # Schalter
    chk_sbert     = W.Checkbox(value=bool(CONFIG.get("USE_SBERT", True) and _HAS_ST), description="SBERT berechnen", disabled=not _HAS_ST)
    chk_bertscore = W.Checkbox(value=bool(CONFIG.get("USE_BERTSCORE", True) and _HAS_BERTSCORE), description="BERTScore berechnen", disabled=not _HAS_BERTSCORE)
    chk_ragas     = W.Checkbox(value=bool(CONFIG.get("USE_RAGAS", True) and _ragas_available), description="RAGAS-Tab anzeigen", disabled=not _ragas_available)
    chk_spacy     = W.Checkbox(value=bool(CONFIG.get("USE_SPACY", True) and _HAS_SPACY), description="spaCy (Lemmata) nutzen", disabled=not _HAS_SPACY)

    # Ampel-Slider
    def _sl(label, key):
        y,g = THRESHOLDS[key]
        return (W.FloatSlider(value=y, min=0, max=1, step=0.01, description=f"{label} gelb"),
                W.FloatSlider(value=g, min=0, max=1, step=0.01, description=f"{label} grün"))

    s_tfidf_y, s_tfidf_g = _sl("TF-IDF", "Cosine (TF-IDF)")
    s_sbert_y, s_sbert_g = _sl("SBERT",  "SBERT Cosine")
    s_bert_y,  s_bert_g  = _sl("BERT F1","BERTScore F1")
    s_rL_y,    s_rL_g    = _sl("ROUGE-L","ROUGE-L F1")
    s_bleu_y,  s_bleu_g  = _sl("BLEU-4", "BLEU-4")

    # Ratio-Bänder & Darstellung
    rng_len = W.FloatRangeSlider(value=LEN_RATIO_BAND, min=0.5, max=1.5, step=0.01, description="len_ratio OK-Band")
    rng_syl = W.FloatRangeSlider(value=SYL_RATIO_BAND, min=0.5, max=1.5, step=0.01, description="syl_ratio OK-Band")
    s_text_w = W.IntSlider(value=UI_TEXT_WIDTH, min=220, max=700, step=10, description="Textbreite (px)")
    s_num_min = W.IntSlider(value=UI_NUM_MIN, min=60, max=150, step=5, description="Num min (px)")
    s_num_max = W.IntSlider(value=UI_NUM_MAX, min=70, max=180, step=5, description="Num max (px)")

    btn_apply = W.Button(description="Übernehmen & neu rendern", button_style="primary", icon="check")
    btn_export = W.Button(description="Auto-Report exportieren (Markdown)", icon="download")
    btn_export_txt = W.Button(description="Vergleich als .txt (TSV) exportieren", icon="download")
    toast = W.HTML("")

    sec_calc = W.VBox([lib_row, W.HBox([chk_sbert, chk_bertscore, chk_ragas, chk_spacy])])
    sec_thr  = W.VBox([s_tfidf_y, s_tfidf_g, s_sbert_y, s_sbert_g, s_bert_y, s_bert_g, s_rL_y, s_rL_g, s_bleu_y, s_bleu_g, W.HTML("<hr>"), rng_len, rng_syl])
    sec_disp = W.VBox([W.HBox([s_text_w, s_num_min, s_num_max])])
    sec_act  = W.VBox([W.HBox([btn_apply, btn_export, btn_export_txt]), toast])

    acc = W.Accordion(children=[sec_calc, sec_thr, sec_disp, sec_act])
    for i, title in enumerate(["Berechnung", "Ampel-Schwellen", "Darstellung", "Aktionen"]):
        acc.set_title(i, title)

    def _notify(msg, ok=True):
        color = "#d4edda" if ok else "#fdecea"
        border = "#28a745" if ok else "#c62828"
        t = datetime.now().strftime("%H:%M:%S")
        toast.value = f"<div style='margin-top:8px;padding:8px 10px;border:1px solid {border};border-radius:6px;background:{color}'>\
                        <b>{'✓' if ok else '!'}</b> {msg} <span style='opacity:.7'>(um {t})</span></div>"

    def on_apply(_):
        global LEN_RATIO_BAND, SYL_RATIO_BAND, UI_TEXT_WIDTH, UI_NUM_MIN, UI_NUM_MAX
        # Flags
        globals()["__COMPUTE_SBERT__"]     = bool(chk_sbert.value)
        globals()["__COMPUTE_BERTSCORE__"] = bool(chk_bertscore.value)
        globals()["__SHOW_RAGAS__"]        = bool(chk_ragas.value)
        CONFIG["USE_SPACY"]                = bool(chk_spacy.value)

        # Schwellen
        THRESHOLDS.update({
            "Cosine (TF-IDF)": (s_tfidf_y.value, s_tfidf_g.value),
            "SBERT Cosine":    (s_sbert_y.value, s_sbert_g.value),
            "BERTScore F1":    (s_bert_y.value,  s_bert_g.value),
            "ROUGE-L F1":      (s_rL_y.value,    s_rL_g.value),
            "BLEU-4":          (s_bleu_y.value,  s_bleu_g.value),
        })
        LEN_RATIO_BAND = tuple(rng_len.value)
        SYL_RATIO_BAND = tuple(rng_syl.value)
        UI_TEXT_WIDTH, UI_NUM_MIN, UI_NUM_MAX = s_text_w.value, s_num_min.value, s_num_max.value

        try:
            rebuild_tabs()
            _notify("Einstellungen übernommen und Tabs neu gerendert.")
            _scroll_top()
        except Exception as e:
            _notify(f"Einstellungen übernommen, aber Rebuild-Hinweis: {e}", ok=False)

    def on_export(_):
        try:
            ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
            path = f"/mnt/data/ad_report_{ts}.md"
            with open(path, "w", encoding="utf-8") as f:
                f.write(report)
            _notify(f"Report gespeichert: <a href='sandbox:{path}' target='_blank'>{os.path.basename(path)}</a>")
            _scroll_top()
        except Exception as e:
            _notify(f"Export fehlgeschlagen: {e}", ok=False)

    def on_export_txt(_):
        try:
            import pandas as pd
            if not (globals().get("__HAVE_DATA__") and isinstance(globals().get("table"), pd.DataFrame)):
                raise RuntimeError("Keine Ergebnisse – bitte zuerst Berechnen.")
    
            base = globals()["table"].copy()
            rag = globals().get("ragas_seg", None)
            if isinstance(rag, pd.DataFrame) and len(rag) == len(base):
                base = pd.concat([base, rag], axis=1)
    
            # RAGAS Anzeige-Header zurück auf Basisnamen
            inv = {v: k for k, v in globals().get("RAGAS_DISPLAY_MAP", {}).items()}
            if inv: base = base.rename(columns=inv)
    
            export_cols_pref = [
                "Start Ref","Ende Ref","Referenzsegment (MDR)",
                "Start Hyp","Ende Hyp","Hypothesensegment (KI)",
                "Cosine (TF-IDF)","SBERT Cosine","BERTScore F1","ROUGE-L F1","BLEU-4",
                "ROUGE-1 P","ROUGE-1 R","ROUGE-2 P","ROUGE-2 R","ROUGE-L P","ROUGE-L R",
                "Wörter Ref","Wörter Hyp","len_ratio","Silben Ref","Silben Hyp","syl_ratio",
                "Ref Farbdetail","Hyp Farbdetail","Ref Bewegung","Hyp Bewegung",
                "RAGAS: semantic_similarity","RAGAS: answer_similarity","RAGAS: faithfulness",
                "RAGAS: answer_relevancy","RAGAS: coverage","RAGAS: conciseness","RAGAS: fluency(FRE)"
            ]
            export_cols = [c for c in export_cols_pref if c in base.columns]
            export_df = base[export_cols].copy() if export_cols else base.copy()
    
            out_txt, out_tsv16 = _export_txt_safely(export_df)
            link = f"<a href='sandbox:{out_txt}' target='_blank'>{os.path.basename(out_txt)}</a>"
            link2 = f" · Excel: <a href='sandbox:{out_tsv16}' target='_blank'>{os.path.basename(out_tsv16)}</a>" if out_tsv16 else ""
            _notify(f"TXT gespeichert: {link}{link2}")
            _scroll_top()
        except Exception as e:
            _notify(f"TXT-Export fehlgeschlagen: {e}", ok=False)

    btn_apply.on_click(on_apply)
    btn_export.on_click(on_export)
    btn_export_txt.on_click(on_export_txt)
    return acc

settings_widget = build_settings_panel()

# =======================
# 🎨 Tabs (farbig) & Tabellen-Styling
# =======================
display(HTML("""
<style>
.widget-tab .p-TabBar-content { padding: 2px 2px 0 2px; }
.widget-tab .p-TabBar-tab {
  margin-right: 6px; border: 1px solid #d7dbe0; border-bottom: none;
  border-radius: 8px 8px 0 0; padding: 6px 12px; font-size: 12px;
  background: #f6f7f9; color: #222;
}
.widget-tab .p-TabBar-tab:hover { filter: brightness(0.98); }
.widget-tab .p-TabBar-tab.p-mod-current { font-weight: 600; box-shadow: 0 -1px 0 0 #999 inset; }
.widget-tab .p-TabBar-tab:nth-child(1) { background:#E3F2FD; }
.widget-tab .p-TabBar-tab:nth-child(2) { background:#E8F5E9; }
.widget-tab .p-TabBar-tab:nth-child(3) { background:#FFF3E0; }
.widget-tab .p-TabBar-tab:nth-child(4) { background:#E0F7FA; }
.widget-tab .p-TabBar-tab:nth-child(5) { background:#F3E5F5; }
.widget-tab .p-TabBar-tab:nth-child(6) { background:#ECEFF1; }
.widget-tab .p-TabBar-tab:nth-child(7) { background:#FFFDE7; }
.widget-tab .p-TabBar-tab:nth-child(8) { background:#F1F8E9; }
</style>
"""))

text_cols = ["Referenzsegment (MDR)", "Hypothesensegment (KI)"]

def _make_view(cols_right):
    cols = text_cols + [c for c in cols_right if c in table.columns]
    return table[cols].copy()

def _style_table(
    df_view,
    text_width_px=300,
    num_min_px=85, num_max_px=100,
    font_px=11, cell_pad="6px 8px",
    wrap_header_prefixes=("ROUGE-","Silben","Wörter","RAGAS"),
    wrap_header_width_px=78,
    col_widths: Dict[str, int] | None = None
):
    text_cols_local = ["Referenzsegment (MDR)", "Hypothesensegment (KI)"]
    num_cols_local  = [c for c in df_view.columns if c not in text_cols_local]

    # --- WICHTIG: sicherstellen, dass Zahlen wirklich Zahlen sind ---
    for c in num_cols_local:
        df_view[c] = pd.to_numeric(df_view[c], errors="coerce")

    float_cols = [c for c in num_cols_local if pd.api.types.is_float_dtype(df_view[c])]
    int_cols   = [c for c in num_cols_local if pd.api.types.is_integer_dtype(df_view[c])]
    fmt_map = {c:"{:.2f}" for c in float_cols}
    for c in int_cols: fmt_map[c] = "{:.0f}"
    for c in ["Wörter Ref","Wörter Hyp","Silben Ref","Silben Hyp"]:
        if c in df_view.columns: fmt_map[c] = "{:.0f}"

    base_styles = [
        # ← Kopfzeile sticky & deckend, damit nix „darunter“ scrollt
        {"selector":"thead th","props":[
            ("position","sticky"),
            ("top","0"),
            ("z-index","3"),
            ("background","#f7f9fc"),
            ("border-bottom","2px solid #e0e0e0"),
            ("box-shadow","0 2px 2px rgba(0,0,0,0.05)"),
            ("text-align","right"),
        ]},
        {"selector":"th","props":[("font-size",f"{font_px}px"),("padding",cell_pad),
                                  ("white-space","normal"),("vertical-align","bottom")]},
        {"selector":"td","props":[("font-size",f"{font_px}px"),("padding",cell_pad),
                                  ("vertical-align","top")]},
        {"selector":"tbody tr:nth-child(odd)", "props":[("background","#f7f9fc")]},
        {"selector":"tbody tr:nth-child(even)","props":[("background","#ffffff")]},
    ]

    # Text-Header linksbündig lassen
    for col_name in text_cols_local:
        if col_name in df_view.columns:
            i = df_view.columns.get_loc(col_name)
            base_styles.append({
                "selector": f"th.col_heading.level0.col{i}",
                "props":[("text-align","left")]
            })

    # Header-Umbruch für ROUGE / RAGAS / Wörter / Silben
    for i, name in enumerate(df_view.columns):
        if any(_norm_col_key(name).startswith(p) for p in ("ROUGE-", "RAGAS")) or name in ("Silben Ref","Silben Hyp","Wörter Ref","Wörter Hyp"):
            base_styles.append({
                "selector": f"th.col_heading.level0.col{i}",
                "props":[
                    ("max-width", f"{wrap_header_width_px}px"),
                    ("white-space","pre-line"),
                    ("word-break","break-word"),
                    ("overflow-wrap","anywhere"),
                    ("line-height","1.15")
                ]
            })

    s = (df_view.style
         .format(fmt_map, na_rep="")
         .set_table_styles(base_styles)
         .set_table_attributes('style="width:100%;"')
         .set_properties(subset=text_cols_local, **{
             "min-width": f"{text_width_px}px", "max-width": f"{text_width_px+120}px",
             "white-space":"normal", "word-break":"keep-all", "text-align":"left",
         })
         .set_properties(subset=num_cols_local, **{
             "min-width": f"{num_min_px}px", "max-width": f"{num_max_px}px",
             "white-space":"nowrap", "text-align":"right",
         }))

    if col_widths:
        for col, w in col_widths.items():
            if col in df_view.columns:
                s = s.set_properties(subset=[col], **{"min-width": f"{w}px","max-width": f"{w}px"})

    def _ampel_series(series):
        col = series.name
        return [f"background-color: {_color_for_kpi(col, v)}" for v in series]

    for c in list(num_cols_local) + [x for x in ["len_ratio","syl_ratio"] if x in df_view.columns]:
        if c in df_view.columns:
            s = s.apply(_ampel_series, axis=0, subset=[c])

    for c in [x for x in ["Hyp Farbdetail","Hyp Bewegung"] if x in df_view.columns]:
        s = s.apply(lambda ser: [f"background-color: {COLOR_OK if bool(v) else COLOR_BAD}"
                                 if pd.notna(v) else "" for v in ser],
                    axis=0, subset=[c])
    try: s = s.hide(axis="index")
    except: 
        try: s = s.hide_index()
        except: pass
    return s

# KPI-Gruppen
group_quality = ["Cosine (TF-IDF)", "SBERT Cosine", "BERTScore F1","ROUGE-L F1","BLEU-4"]
group_rouge   = ["ROUGE-L P","ROUGE-L R","ROUGE-2 P","ROUGE-2 R","ROUGE-1 P","ROUGE-1 R"]
group_length  = ["Wörter Ref","Wörter Hyp","len_ratio","Silben Ref","Silben Hyp","syl_ratio",
                 "Ref Farbdetail","Hyp Farbdetail","Ref Bewegung","Hyp Bewegung"]
group_align   = ["Ref-Index","Hyp-Index"]

# Breiten-Mapping (gezielt)
rouge_widths = {"ROUGE-L P":70,"ROUGE-L R":70,"ROUGE-2 P":70,"ROUGE-2 R":70,"ROUGE-1 P":70,"ROUGE-1 R":70}
length_widths = {"Wörter Ref":70,"Wörter Hyp":70,"len_ratio":70,"Silben Ref":80,"Silben Hyp":80,"syl_ratio":70,
                 "Ref Farbdetail":80,"Hyp Farbdetail":80,"Ref Bewegung":80,"Hyp Bewegung":80}

def _text_grid_html(pairs, ref_segs, hyp_segs):
    # Kein Alignment? – kurze Info zurück
    if not isinstance(pairs, (list, tuple)) or len(pairs) == 0:
        return """
        <div class='adgrid'>
          <p><em>Noch keine Ausrichtung vorhanden. Bitte oben <b>Berechnen</b> ausführen.</em></p>
        </div>
        """

    rows_html = []
    for tup in pairs:
        try:
            r, h, _ = tup
            if 0 <= r < len(ref_segs) and 0 <= h < len(hyp_segs):
                rs, hs = escape(ref_segs[r]), escape(hyp_segs[h])
                rows_html.append(f"<tr><td class='ref'>{rs}</td><td class='hyp'>{hs}</td></tr>")
        except Exception:
            continue

    if not rows_html:
        return """
        <div class='adgrid'>
          <p><em>Keine gültigen Paare gefunden. Bitte erneut <b>Berechnen</b> ausführen.</em></p>
        </div>
        """

    return f"""
    <style>
      /* eigener Scroll-Container NUR für die Texttabelle */
      .adgrid-scroll {{
        max-height: 60vh;                 /* Höhe nach Bedarf */
        overflow-y: auto;                 /* <— Scrollen passiert hier */
        overflow-x: hidden;
        border: 1px solid #e6e9ef;
        border-radius: 8px;
        background: #fff;
      }}
      .adgrid table {{ width:100%; border-collapse:separate; border-spacing:0 6px; }}
      .adgrid th {{ font-weight:600; padding:6px 8px; text-align:left; }}
      .adgrid td {{ vertical-align:top; padding:6px 8px; line-height:1.35; }}
      .adgrid td.ref, .adgrid td.hyp {{ width:50%; }}
      .adgrid tbody tr:nth-child(odd) td {{ background:#f0f7ff; }}

      /* sticky Header relativ zum .adgrid-scroll-Container */
      .adgrid thead th {{
        position: sticky;
        top: 0;                           /* klebt ganz oben im Scroll-Container */
        z-index: 2;
        background: #f7f9fc;              /* sichtbar über den Zeilen */
        border-bottom: 2px solid #e0e0e0;
        box-shadow: 0 2px 2px rgba(0,0,0,0.05);
      }}
    </style>

    <div class="adgrid">
      <div class="adgrid-scroll">
        <table>
          <thead>
            <tr><th>Referenzsegment (MDR)</th><th>Hypothesensegment (KI)</th></tr>
          </thead>
          <tbody>
            {''.join(rows_html)}
          </tbody>
        </table>
      </div>
    </div>
    """

def rebuild_tabs():
    text_html = _text_grid_html(pairs, ref_segs, hyp_segs)

    view_quality = _style_table(_make_view(group_quality))
    view_rouge   = _style_table(_make_view(group_rouge), text_width_px=280, num_min_px=70, num_max_px=90,
                                wrap_header_prefixes=("ROUGE-",), wrap_header_width_px=65, col_widths=rouge_widths)
    view_length  = _style_table(_make_view(group_length), text_width_px=280, num_min_px=70, num_max_px=95,
                                col_widths=length_widths)
    view_align   = _style_table(_make_view(group_align))

    # "Alle" (inkl. RAGAS-Kolumnen mit Anzeige-Header)
    if isinstance(globals().get("ragas_seg"), pd.DataFrame) and len(ragas_seg) == len(table):
        table_all = pd.concat([table, ragas_seg], axis=1)
    else:
        table_all = table.copy()
    table_all_disp = table_all.rename(columns=RAGAS_DISPLAY_MAP)
    all_order = (
        text_cols
        + ["Cosine (TF-IDF)", "SBERT Cosine", "BERTScore F1", "ROUGE-L F1", "BLEU-4",
           "ROUGE-L P","ROUGE-L R","ROUGE-2 P","ROUGE-2 R","ROUGE-1 P","ROUGE-1 R"]
        + ["Wörter Ref","Wörter Hyp","Silben Ref","Silben Hyp","len_ratio","syl_ratio",
           "Ref Farbdetail","Hyp Farbdetail","Ref Bewegung","Hyp Bewegung"]
        + ["Ref-Index","Hyp-Index"]
        + ["RAGAS:\nsemantic\nsimilarity","RAGAS:\nanswer\nsimilarity","RAGAS:\nfaithfulness",
           "RAGAS:\nanswer\nrelevancy","RAGAS:\ncoverage","RAGAS:\nconciseness","RAGAS:\nfluency\n(FRE)"]
    )
    all_order = [c for c in all_order if c in table_all_disp.columns]
    table_all_disp = table_all_disp[all_order]

    col_widths_all = {}
    col_widths_all.update(length_widths)
    col_widths_all.update({
        "RAGAS:\nsemantic\nsimilarity": 92,
        "RAGAS:\nanswer\nsimilarity":   92,
        "RAGAS:\nfaithfulness":         92,
        "RAGAS:\nanswer\nrelevancy":    92,
        "RAGAS:\ncoverage":             92,
        "RAGAS:\nconciseness":          92,
        "RAGAS:\nfluency\n(FRE)":       96,
    })

    view_all = _style_table(
        table_all_disp,
        text_width_px=540,
        num_min_px=85, num_max_px=110,
        font_px=12,
        wrap_header_prefixes=("ROUGE-", "RAGAS"),
        wrap_header_width_px=90,
        col_widths=col_widths_all
    )

    # RAGAS (eigener Tab)
    ragas_view = pd.concat([table[text_cols], ragas_seg], axis=1).rename(columns=RAGAS_DISPLAY_MAP)
    ragas_widths = {
        "RAGAS:\nsemantic\nsimilarity":  92,
        "RAGAS:\nanswer\nsimilarity":    92,
        "RAGAS:\nfaithfulness":          92,
        "RAGAS:\nanswer\nrelevancy":     92,
        "RAGAS:\ncoverage":              92,
        "RAGAS:\nconciseness":           92,
        "RAGAS:\nfluency\n(FRE)":        96,
    }
    view_ragas = _style_table(
        ragas_view,
        text_width_px=420,
        num_min_px=70, num_max_px=95,
        font_px=12,
        wrap_header_prefixes=("RAGAS",),
        wrap_header_width_px=90,
        col_widths=ragas_widths
    )

    # Status
    cfg_rows = [
        ("Sprache", CONFIG["LANG"]),
        ("spaCy aktiviert", str(_HAS_SPACY and CONFIG.get("USE_SPACY", True))),
        ("SBERT aktiviert", str(_HAS_ST and __COMPUTE_SBERT__)),
        ("BERTScore aktiviert", str(_HAS_BERTSCORE and __COMPUTE_BERTSCORE__)),
        ("RAGAS verfügbar", str(_ragas_available)),
        ("Silben-Methode", CONFIG["SYLLABLE_METHOD"]),
    ]
    cfg_table = pd.DataFrame(cfg_rows, columns=["Einstellung","Wert"])
    view_settings = (cfg_table.style
                     .hide(axis="index")
                     .set_table_styles([{"selector":"th","props":[("text-align","left"),("font-size","12px")]},
                                        {"selector":"td","props":[("font-size","12px")]}]))

    # Tabs rendern – Charts + Kennzahlen im 1. Tab (Text)
    with tabs_out:
        tabs_out.clear_output()
        import ipywidgets as widgets

        # Titel-Liste ANLEGEN (sonst NameError)
        titles = ["Text", "Qualität", "ROUGE", "Länge & Abdeckung", "Alignment"]

        text_tab = widgets.VBox(
            [widgets.HTML(text_html), charts_out, summary_out],
            layout=widgets.Layout(overflow_y='visible')
        )

        children = [
            text_tab,
            widgets.HTML(sticky_styler(view_quality)),
            widgets.HTML(sticky_styler(view_rouge)),
            widgets.HTML(sticky_styler(view_length)),
            widgets.HTML(sticky_styler(view_align)),
        ]

        if globals().get("__SHOW_RAGAS__", True):
            children.append(widgets.HTML(sticky_styler(view_ragas)))
            titles.append("RAGAS")

        children.append(widgets.HTML(sticky_styler(view_all)))
        titles.append("Alle")

        children.append(settings_widget)
        titles.append("⚙️ Einstellungen")

        tabs = widgets.Tab(children=children)
        tabs.add_class('ad-tabs')
        tabs.layout = widgets.Layout(max_height='72vh', overflow_y='auto', overflow_x='hidden')

        for i, t in enumerate(titles):
            tabs.set_title(i, t)

        display(tabs)
        
# =======================
# 📈 Visualisierungen – nur rendern, wenn Daten vorhanden
# =======================
ROW_FIGSIZE = (16, 5.2)
TITLE_SIZE  = 16
LABEL_SIZE  = 13
TICK_SIZE   = 12
GRID_ALPHA  = 0.25
DOT_SIZE    = 42
LINEWIDTH   = 1.6

plt.rcParams.update({
    "axes.grid": True,
    "grid.alpha": GRID_ALPHA,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.labelsize": LABEL_SIZE,
    "axes.titlesize": TITLE_SIZE,
    "xtick.labelsize": TICK_SIZE,
    "ytick.labelsize": TICK_SIZE,
})

def _nice_bins(vals):
    vals = np.asarray(vals, dtype=float)
    vals = vals[~np.isnan(vals)]
    if len(vals) <= 1: return 5
    q75, q25 = np.percentile(vals, [75, 25])
    iqr = max(q75 - q25, 1e-9)
    binw = 2 * iqr / (len(vals) ** (1/3))
    bins = int(np.ceil((vals.max() - vals.min()) / max(binw, 1e-12)))
    return int(np.clip(bins, 5, 20))

def _fix_01_axis(ax, vals):
    vmin = np.nanmin(vals); vmax = np.nanmax(vals)
    if vmin >= -0.05 and vmax <= 1.05: ax.set_xlim(0, 1)

def _annotate_mean(ax, mean_val):
    ymax = ax.get_ylim()[1]
    ax.axvline(mean_val, linestyle="--", linewidth=LINEWIDTH)
    ax.text(mean_val, ymax*0.95, f"μ = {mean_val:.2f}",
            ha="right", va="top",
            bbox=dict(facecolor="white", alpha=0.85, boxstyle="round,pad=0.2"))

def _hist(ax, series, title, xlabel):
    vals = np.asarray(series, dtype=float); vals = vals[~np.isnan(vals)]
    if len(vals) == 0:
        ax.set_axis_off(); ax.text(0.5, 0.5, "Keine Daten", ha="center", va="center", fontsize=LABEL_SIZE); return
    ax.hist(vals, bins=_nice_bins(vals), edgecolor="white")
    _fix_01_axis(ax, vals); _annotate_mean(ax, float(np.mean(vals)))
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel("Häufigkeit")

def _pair_rows(panels):
    for i in range(0, len(panels), 2):
        fig, axes = plt.subplots(1, 2, figsize=ROW_FIGSIZE, constrained_layout=True)
        panels[i](axes[0])
        if i+1 < len(panels): panels[i+1](axes[1])
        else: axes[1].set_visible(False)
        plt.show()

def draw_charts():
    """Charts nur zeichnen, wenn bereits berechnet wurde."""
    plt.close('all')
    if not (globals().get("__HAVE_DATA__") and "df" in globals() and isinstance(df, pd.DataFrame) and not df.empty):
        display(HTML("<em>Noch keine Charts – bitte zuerst <b>Berechnen</b>.</em>"))
        return

    panels = []
    panels.append(lambda ax: _hist(ax, df["alignment_cosine"], "TF-IDF Cosine – Verteilung", "Cosine"))
    panels.append(lambda ax: _hist(ax, df["rougeL_f1"],        "ROUGE-L F1 – Verteilung",    "ROUGE-L F1"))
    if "bertscore_f1" in df:  panels.append(lambda ax: _hist(ax, df["bertscore_f1"], "BERTScore F1 – Verteilung", "BERTScore F1"))
    if "sbert_cosine" in df:  panels.append(lambda ax: _hist(ax, df["sbert_cosine"], "SBERT Cosine – Verteilung", "Cosine"))
    panels.append(lambda ax: _hist(ax, df["bleu_4"],           "BLEU-4 – Verteilung",        "BLEU-4"))
    panels.append(lambda ax: _hist(ax, table["len_ratio"],     "Längenverhältnis (Wörter) – Verteilung", "len_hyp / len_ref"))
    panels.append(lambda ax: _hist(ax, table["syl_ratio"],     "Längenverhältnis (Silben) – Verteilung", "syl_hyp / syl_ref"))

    if globals().get("__SHOW_RAGAS__") and "ragas_seg" in globals():
        for col, title in [
            ("RAGAS: semantic_similarity", "RAGAS: Semantic Similarity"),
            ("RAGAS: faithfulness",        "RAGAS: Faithfulness"),
            ("RAGAS: answer_relevancy",    "RAGAS: Answer Relevancy"),
        ]:
            if col in ragas_seg:
                panels.append(lambda ax, c=col, t=title: _hist(ax, ragas_seg[c], t, "Score"))

    _pair_rows(panels)

# =======================
# 🧮 Aggregation + 🧠 Rubrics + 🧾 Report – nur wenn Daten da
# =======================
def render_summary_and_report():
    """Aggregationen, Rubrics & Auto-Report nur mit Daten rendern."""
    if not (globals().get("__HAVE_DATA__") and "df" in globals() and isinstance(df, pd.DataFrame) and not df.empty):
        return

    # --- Aggregation
    metric_cols = [c for c in df.columns if any(
        c.startswith(p) for p in ["alignment_cosine","sbert","bertscore","rouge","bleu","len_","syl_"]
    )]
    summary = pd.DataFrame({
        "n":          df[metric_cols].count(),
        "Mittelwert": df[metric_cols].mean(numeric_only=True),
        "StdAbw":     df[metric_cols].std(numeric_only=True, ddof=0),
        "Median":     df[metric_cols].median(numeric_only=True),
        "Min":        df[metric_cols].min(numeric_only=True),
        "Max":        df[metric_cols].max(numeric_only=True),
    })
    summary["CV %"] = np.where(summary["Mittelwert"] > 0,
                               100 * summary["StdAbw"] / summary["Mittelwert"], np.nan)

    rename_map = {
        "alignment_cosine":"Cosine (TF-IDF)",
        "sbert_cosine":"SBERT Cosine",
        "bertscore_f1":"BERTScore F1",
        "rouge1_precision":"ROUGE-1 P","rouge1_recall":"ROUGE-1 R","rouge1_f1":"ROUGE-1 F1",
        "rouge2_precision":"ROUGE-2 P","rouge2_recall":"ROUGE-2 R","rouge2_f1":"ROUGE-2 F1",
        "rougeL_precision":"ROUGE-L P","rougeL_recall":"ROUGE-L R","rougeL_f1":"ROUGE-L F1",
        "bleu_1":"BLEU-1","bleu_2":"BLEU-2","bleu_3":"BLEU-3","bleu_4":"BLEU-4",
        "len_ref":"Wörter Ref","len_hyp":"Wörter Hyp",
        "syl_ref":"Silben Ref","syl_hyp":"Silben Hyp",
    }
    summary = summary.rename(index=rename_map)

    order = [
        "Cosine (TF-IDF)", "SBERT Cosine", "BERTScore F1", "ROUGE-L F1", "BLEU-4",
        "ROUGE-L P","ROUGE-L R","ROUGE-2 P","ROUGE-2 R","ROUGE-1 P","ROUGE-1 R",
        "Wörter Ref","Wörter Hyp","Silben Ref","Silben Hyp",
    ]
    order = [i for i in order if i in summary.index] + [i for i in summary.index if i not in order]
    summary = summary.loc[order]

    styled_summary = (
        summary.style
        .format({"n": "{:.0f}", "Mittelwert": "{:.2f}", "StdAbw": "{:.2f}",
                 "Median": "{:.2f}", "Min": "{:.2f}", "Max": "{:.2f}", "CV %": "{:.0f}"}, na_rep="")
        .set_table_styles([
            {"selector":"th","props":[("font-size","12px"),("text-align","left")]},
            {"selector":"td","props":[("font-size","12px")]}
        ])
        .background_gradient(subset=["Mittelwert"], cmap="Greens")
        .background_gradient(subset=["CV %"], cmap="Reds")
    )

    # --- RAGAS-Summary (safe)
    def _build_ragas_summary_df(ragas_seg_df):
        import pandas as _pd
        if not isinstance(ragas_seg_df, _pd.DataFrame) or ragas_seg_df.empty:
            return _pd.DataFrame(columns=["Mittelwert"])
        m = ragas_seg_df.mean(numeric_only=True).rename("Mittelwert").to_frame()
        try:
            m = m.rename(index=RAGAS_DISPLAY_MAP)
        except Exception:
            pass
        return m

    try:
        ragas_summary = _build_ragas_summary_df(globals().get("ragas_seg", None))
    except Exception:
        import pandas as _pd
        ragas_summary = _pd.DataFrame(columns=["Mittelwert"])

    try:
        _left_html  = styled_summary.to_html()
    except Exception:
        _left_html  = "<em>Keine Zusammenfassung vorhanden.</em>"

    try:
        _right_html = (
            ragas_summary.style.format("{:.2f}").to_html()
            if hasattr(ragas_summary, "style") and not ragas_summary.empty
            else "<em>Keine RAGAS-Ergebnisse vorhanden.</em>"
        )
    except Exception:
        _right_html = "<em>RAGAS-Zusammenfassung nicht verfügbar.</em>"

    display(HTML(f"""
    <div style='display:grid;grid-template-columns:1fr 1fr;gap:16px;align-items:start'>
      <div>{_left_html}</div>
      <div>{_right_html}</div>
    </div>
    """))

    # --- Mikro-Scores (gesamte Texte)
    try:
        ref_all = [t for toks in ref_tok for t in toks]
        hyp_all = [t for toks in hyp_tok for t in toks]
        micro = {}
        micro["BLEU-4 (micro)"] = bleu_score(ref_all, hyp_all, max_n=4)
        micro["ROUGE-L F1 (micro)"] = rouge_l(ref_all, hyp_all)["f1"]
        micro["Cosine (TF-IDF, micro)"] = float(cosine_similarity_matrix(
            *build_tfidf_vectors([ref_all, hyp_all]))[0,1])
        micro_row = pd.DataFrame(micro, index=["Gesamt"])
        display(micro_row.style.format("{:.2f}"))
    except Exception:
        pass

    # --- Rubrics
    def z01(x): 
        return float(np.clip(x, 0, 1)) if pd.notna(x) else np.nan

    def _score_bar_html(v: float) -> str:
        if pd.isna(v): return "—"
        p = int(np.clip(round(float(v)), 0, 100))
        return (
            f"<div style='display:flex;align-items:center;gap:.5rem'>"
            f"  <div style='flex:1;background:#eef5ee;border-radius:10px;height:18px;position:relative;overflow:hidden'>"
            f"    <div style='width:{p}%;height:100%;background:linear-gradient(90deg,#C8E6C9,#81C784)'></div>"
            f"  </div>"
            f"  <div style='min-width:32px;text-align:right;font-variant-numeric:tabular-nums'>{p}</div>"
            f"</div>"
        )

    def _chip_html(v: float) -> str:
        if pd.isna(v): return ""
        s = float(v)
        if s >= 85:
            label, bg, bd, fg = "stark", "#E8F5E9", "#66BB6A", "#2E7D32"
        elif s >= 70:
            label, bg, bd, fg = "ok",    "#FFF8E1", "#FFB300", "#8C6D1F"
        else:
            label, bg, bd, fg = "prüfen","#FFEBEE", "#E57373", "#B71C1C"
        return f"<span style='padding:2px 8px;border:1px solid {bd};border-radius:999px;background:{bg};color:{fg};font-size:12px'>{label}</span>"

    def show_rubrics_pretty(rubrics_df: pd.DataFrame, overall_score: float | int | None = None):
        vis = rubrics_df.copy()
        vis.insert(1, "Score", vis["Score (0–100)"].map(_score_bar_html))
        vis.insert(2, "Bewertung", vis["Score (0–100)"].map(_chip_html))
        vis = vis.drop(columns=["Score (0–100)"])
        base_styles = [
            {"selector":"th","props":[("font-size","13px"),("text-align","left"),
                                      ("padding","6px 10px"),("border-bottom","1px solid #ddd")]},
            {"selector":"td","props":[("font-size","13px"),("padding","6px 10px"),("vertical-align","middle")]},
            {"selector":"tbody tr:nth-child(odd)", "props":[("background","#f7f9fc")]},
            {"selector":"tbody tr:nth-child(even)","props":[("background","#ffffff")]},
        ]
        sty = (vis.style
               .set_table_styles(base_styles)
               .set_properties(subset=["Rubrik","Begründung"], **{"white-space":"normal"})
               .format(na_rep=""))
        try:    sty = sty.hide(axis="index")
        except: 
            try: sty = sty.hide_index()
            except: pass
        display(HTML("<h4 style='margin:6px 0 6px 0'>MDR-Rubrics (0–100, höher besser)</h4>"))
        try:    html = sty.to_html(escape=False)
        except TypeError: html = sty.to_html()
        display(HTML(html))
        if overall_score is not None and not pd.isna(overall_score):
            overall_int = int(round(float(overall_score)))
            display(HTML(
                f"<div style='margin-top:.6rem'><strong>Gesamt (gewichtet):</strong> "
                f"<span style='display:inline-block;margin-left:.35rem;padding:.15rem .6rem;"
                f"border:1px solid #66BB6A;border-radius:999px;background:#E8F5E9;"
                f"color:#2E7D32;font-weight:600'>{overall_int} / 100</span></div>"
            ))

    pct = lambda s: float(np.mean(s)*100.0) if len(s) else np.nan
    ref_color = pct(table["Ref Farbdetail"]) if "Ref Farbdetail" in table else np.nan
    hyp_color = pct(table["Hyp Farbdetail"]) if "Hyp Farbdetail" in table else np.nan
    ref_move  = pct(table["Ref Bewegung"])  if "Ref Bewegung"  in table else np.nan
    hyp_move  = pct(table["Hyp Bewegung"])  if "Hyp Bewegung"  in table else np.nan
    color_gap = max(0.0, (ref_color - hyp_color)/100.0) if (pd.notna(ref_color) and pd.notna(hyp_color)) else np.nan
    move_gap  = max(0.0, (ref_move  - hyp_move )/100.0) if (pd.notna(ref_move)  and pd.notna(hyp_move )) else np.nan

    len_ratio_mean = float(np.nanmean(table["len_ratio"])) if "len_ratio" in table else np.nan
    syl_ratio_mean = float(np.nanmean(table["syl_ratio"])) if "syl_ratio" in table else np.nan
    rougeL_mean    = float(np.nanmean(df["rougeL_f1"]))    if "rougeL_f1" in df   else np.nan
    sbert_mean     = float(np.nanmean(df["sbert_cosine"])) if "sbert_cosine" in df else np.nan

    FRE_hyp = flesch_amstad(" ".join(hyp_segs))
    FRE_score = z01(FRE_hyp/100.0)

    def score_conciseness(r):
        if pd.isna(r): return np.nan
        return float((1 - min(1.0, abs(r-1.0))) * 100)

    def score_clarity(gap):
        if pd.isna(gap): return np.nan
        return float((1 - np.clip(gap, 0, 1)) * 100)

    def score_alignment(rougeL, sbert):
        base = np.nanmean([z01(rougeL), z01(sbert)])
        return float((base if pd.notna(base) else np.nan) * 100)

    def score_readability(fre_norm):
        return float((fre_norm if pd.notna(fre_norm) else np.nan) * 100)

    rows_rub = [
        ["Stil/Konzision (Wörter)" , score_conciseness(len_ratio_mean), "Nähe der Länge (Wörter) zu 1.0"],
        ["Stil/Konzision (Silben)" , score_conciseness(syl_ratio_mean), "Nähe der Länge (Silben) zu 1.0"],
        ["Lesehärte (FRE, Hyp)"    , score_readability(FRE_score)     , "Flesch–Amstad; höher = leichter"],
        ["Visuelle Klarheit"       , score_clarity(color_gap)          , "Gap Farbdetails Hyp vs. Ref (klein ist gut)"],
        ["Handlungsführung"        , score_clarity(move_gap)           , "Gap Bewegungsverben Hyp vs. Ref (klein ist gut)"],
        ["Inhaltsdeckung/Kohärenz" , score_alignment(rougeL_mean, sbert_mean),
                                      "ROUGE-L F1 & SBERT Cosine (Mittel)"],
    ]
    RUBRICS = pd.DataFrame(rows_rub, columns=["Rubrik","Score (0–100)","Begründung"])
    RUBRICS["Score (0–100)"] = RUBRICS["Score (0–100)"].clip(0, 100)

    weights = {
        "Stil/Konzision (Wörter)" : 1.0,
        "Stil/Konzision (Silben)" : 1.0,
        "Lesehärte (FRE, Hyp)"    : 1.0,
        "Visuelle Klarheit"       : 1.0,
        "Handlungsführung"        : 1.0,
        "Inhaltsdeckung/Kohärenz" : 1.0,
    }
    num = den = 0.0
    for r in rows_rub:
        rubrik, score = r[0], r[1]
        if pd.notna(score):
            w = weights.get(rubrik, 1.0)
            num += score * w; den += w
    overall = (num / den) if den else np.nan

    show_rubrics_pretty(RUBRICS, overall)

    # --- Report
    def _safe(summary, row, col="Mittelwert"):
        return summary.loc[row, col] if row in summary.index and col in summary.columns else np.nan

    def _level(val, thr):
        if pd.isna(val) or thr is None: return "—"
        y, g = thr
        return "hoch" if val >= g else ("mittel" if val >= y else "niedrig")

    def _stability(cv):
        if pd.isna(cv): return "—"
        return "sehr stabil" if cv <= 20 else ("moderat" if cv <= 40 else "volatil")

    cos_mean = _safe(summary, "Cosine (TF-IDF)"); cos_cv = _safe(summary, "Cosine (TF-IDF)", "CV %")
    sbert_m  = _safe(summary, "SBERT Cosine");    sbert_cv = _safe(summary, "SBERT Cosine", "CV %")
    bert_m   = _safe(summary, "BERTScore F1");    bert_cv  = _safe(summary, "BERTScore F1", "CV %")
    rL_m     = _safe(summary, "ROUGE-L F1");      rL_cv    = _safe(summary, "ROUGE-L F1", "CV %")
    bleu4_m  = _safe(summary, "BLEU-4");          bleu4_cv = _safe(summary, "BLEU-4", "CV %")

    w_ref = _safe(summary, "Wörter Ref"); w_hyp = _safe(summary, "Wörter Hyp")
    s_ref = _safe(summary, "Silben Ref"); s_hyp = _safe(summary, "Silben Hyp")
    len_ratio = (w_hyp / w_ref) if (pd.notna(w_ref) and w_ref > 0) else np.nan
    syl_ratio = (s_hyp / s_ref) if (pd.notna(s_ref) and s_ref > 0) else np.nan

    ref_color_pct = pct(table["Ref Farbdetail"]) if "Ref Farbdetail" in table else np.nan
    hyp_color_pct = pct(table["Hyp Farbdetail"]) if "Hyp Farbdetail" in table else np.nan
    ref_move_pct  = pct(table["Ref Bewegung"])  if "Ref Bewegung"  in table else np.nan
    hyp_move_pct  = pct(table["Hyp Bewegung"])  if "Hyp Bewegung"  in table else np.nan

    def mean_or_nan(col):
        return float(np.nanmean(ragas_seg[col])) if col in ragas_seg else np.nan
    rg_sem = mean_or_nan("RAGAS: semantic_similarity")
    rg_fai = mean_or_nan("RAGAS: faithfulness")
    rg_rel = mean_or_nan("RAGAS: answer_relevancy")
    rg_cov = mean_or_nan("RAGAS: coverage")
    rg_con = mean_or_nan("RAGAS: conciseness")
    rg_flu = mean_or_nan("RAGAS: fluency(FRE)")

    recs = []
    if pd.notna(len_ratio):
        if len_ratio < 0.9:  recs.append("Hyp ist deutlich **kürzer** als Ref → ggf. fehlende Inhalte ergänzen.")
        if len_ratio > 1.1:  recs.append("Hyp ist **länger** als Ref → unnötige Ausschmückungen kürzen.")
    if pd.notna(syl_ratio):
        if syl_ratio > 1.1 and FRE_hyp < 60:
            recs.append("**Viele Silben** & **niedriger FRE** → kürzere Wörter/Strukturen nutzen (leichtere Sprache).")
    if pd.notna(hyp_color_pct) and pd.notna(ref_color_pct) and hyp_color_pct + 5 < ref_color_pct:
        recs.append("**Farbdetails** seltener als in der Ref → visuelle Spezifika konkreter nennen.")
    if pd.notna(hyp_move_pct) and pd.notna(ref_move_pct) and hyp_move_pct + 5 < ref_move_pct:
        recs.append("**Bewegungsverben** seltener als in der Ref → Handlungen klarer benennen.")
    if pd.notna(bleu4_m) and pd.notna(bert_m) and bleu4_m < 0.20 and bert_m >= 0.85:
        recs.append("**BLEU-4 niedrig**, **BERTScore hoch** → starke Paraphrasen (Inhalt ähnlich, Wortlaut anders).")
    if pd.notna(rL_m) and rL_m < 0.50:
        recs.append("**ROUGE-L F1 < 0.50** → Reihenfolge/Abdeckung prüfen; ggf. Segmente neu ausrichten.")
    if (pd.notna(cos_cv) and cos_cv > 40) or (pd.notna(rL_cv) and rL_cv > 40):
        recs.append("Hohe **Streuung** (CV%) → Qualität schwankt; schwache Segmente gezielt verbessern.")
    if pd.notna(rg_fai) and rg_fai < 0.6:
        recs.append("**Faithfulness** gering → wichtige Referenzinhalte fehlen oder sind entstellt.")
    if pd.notna(rg_rel) and rg_rel < 0.6:
        recs.append("**Relevancy** gering → Hyp enthält Zusatz-/Nebenaspekte; stärker auf Relevantes fokussieren.")
    if not recs:
        recs = ["Keine akuten Auffälligkeiten. Feinjustierung nach Bedarf."]

    def _fmt0(x):
        return "—" if (x is None or (isinstance(x, float) and np.isnan(x))) else f"{float(x):.0f}"

    rubrics_vals = {r: s for r, s, _ in rows_rub}
    rubrics_md = (
        f"- **Stil/Konzision (Wörter):** {_fmt0(rubrics_vals.get('Stil/Konzision (Wörter)'))}/100 – Nähe der Länge (Wörter) zu 1.0\n"
        f"- **Stil/Konzision (Silben):** {_fmt0(rubrics_vals.get('Stil/Konzision (Silben)'))}/100 – Nähe der Länge (Silben) zu 1.0\n"
        f"- **Lesehärte (FRE, Hyp):** {_fmt0(rubrics_vals.get('Lesehärte (FRE, Hyp)'))}/100 – höher = leichter\n"
        f"- **Visuelle Klarheit:** {_fmt0(rubrics_vals.get('Visuelle Klarheit'))}/100 – Gap Farbdetails (kleiner ist besser)\n"
        f"- **Handlungsführung:** {_fmt0(rubrics_vals.get('Handlungsführung'))}/100 – Gap Bewegungsverben (kleiner ist besser)\n"
        f"- **Inhaltsdeckung/Kohärenz:** {_fmt0(rubrics_vals.get('Inhaltsdeckung/Kohärenz'))}/100 – ROUGE-L F1 & SBERT (Mittel)\n"
        f"- **Gesamtnote (gewichtet):** {_fmt0(overall)}/100"
    )

    report = f"""
# Automatische Interpretation (mit RAGAS & Rubrics)

**Gesamtqualität (Mittelwerte)**  
- TF-IDF Cosine: **{cos_mean:.2f}** ({_level(cos_mean, THRESHOLDS.get('Cosine (TF-IDF)'))}), Stabilität: {_stability(cos_cv)}  
- SBERT Cosine: **{sbert_m:.2f}** ({_level(sbert_m, THRESHOLDS.get('SBERT Cosine'))}), Stabilität: {_stability(sbert_cv)}  
- BERTScore F1: **{bert_m:.2f}** ({_level(bert_m, THRESHOLDS.get('BERTScore F1'))}), Stabilität: {_stability(bert_cv)}  
- ROUGE-L F1: **{rL_m:.2f}** ({_level(rL_m, THRESHOLDS.get('ROUGE-L F1'))}), Stabilität: {_stability(rL_cv)}  
- BLEU-4: **{bleu4_m:.2f}** ({_level(bleu4_m, THRESHOLDS.get('BLEU-4'))}), Stabilität: {_stability(bleu4_cv)}

**Länge & Lesehärte**  
- Wörter (Ref vs. Hyp): **{w_ref:.0f}** vs. **{w_hyp:.0f}**  → Verhältnis **{(len_ratio if pd.notna(len_ratio) else np.nan):.2f}**  
- Silben  (Ref vs. Hyp): **{s_ref:.0f}** vs. **{s_hyp:.0f}**  → Verhältnis **{(syl_ratio if pd.notna(syl_ratio) else np.nan):.2f}**  
- Lesbarkeit Hyp (Flesch–Amstad): **{FRE_hyp:.0f}**/100 (höher = leichter)

**RAGAS (Fallback-äquivalent, 0..1)**  
- Semantic Similarity: **{rg_sem:.2f}** · Faithfulness: **{rg_fai:.2f}** · Answer Relevancy: **{rg_rel:.2f}**  
- Coverage: **{rg_cov:.2f}** · Conciseness: **{rg_con:.2f}** · Fluency (aus FRE): **{rg_flu:.2f}**

**Heuristiken (Anteil Segmente mit Merkmal)**  
- Farbdetail: Ref **{(ref_color_pct if not pd.isna(ref_color_pct) else 0):.0f}%**, Hyp **{(hyp_color_pct if not pd.isna(hyp_color_pct) else 0):.0f}%**  
- Bewegung:  Ref **{(ref_move_pct  if not pd.isna(ref_move_pct)  else 0):.0f}%**, Hyp **{(hyp_move_pct  if not pd.isna(hyp_move_pct)  else 0):.0f}%**

**MDR-Rubrics (0–100, höher besser)**  
{rubrics_md}

**Empfehlungen**  
- {("\n- ").join(recs)}
"""
    # Für Export-Button verfügbar machen
    globals()["report"] = report
    display(Markdown(report))